# Pipeline Pra-Pemrosesan Data Uji — Sistem Kepatuhan Proposal Kegiatan Mahasiswa (RAG)

Notebook ini merangkum **seluruh Bagian A–G** modul pra-pemrosesan (Python) yang aslinya tersusun sebagai
paket modular `comply_proposal.preprocessing` di repo GitHub, digabung jadi **satu file** supaya gampang
dijalankan langsung di Google Colab tanpa perlu `pip install` paket kustom atau mengelola banyak file `.py`.

**Konteks sistem**: LLM Qwen3-8B (self-hosted, tanpa fine-tuning), embedding BGE-M3, vector DB FAISS,
chunking KB berbasis struktur dokumen (Bab/Pasal/ayat), deployment HuggingFace Spaces (upload PDF & DOCX).
Unit evaluasi = 16 unit fungsional proposal (U00–U15), bukan chunk teknis mentah.

## Struktur notebook
| Bagian | Isi | Fungsi utama |
|---|---|---|
| A | Ekstraksi teks (jalur JP/JC) | `ekstrak_docx`, `ekstrak_pdf`, `ekstrak_dokumen` |
| B | Segmentasi ke 16 unit anotasi | `segmentasi_unit` |
| C | Metadata tata letak (jalur JM) | `ekstrak_metadata_docx`, `ekstrak_metadata_pdf`, `ekstrak_metadata_dokumen` |
| D | Pemetaan ground truth (skema & kode dikonfirmasi dari workbook asli) | `muat_ground_truth`, `gabungkan_teks_dan_ground_truth` |
| E | Statistik dataset | `hitung_statistik` |
| F | Split dev/test | `split_dataset` |
| G | Orkestrasi pipeline & output akhir | `proses_folder_proposal`, `tulis_output_jsonl`, `tulis_log_audit` |

Jalankan sel-sel **dari atas ke bawah secara berurutan** (Bagian A harus dieksekusi sebelum B, dst.,
karena Bagian B–G memanggil fungsi dari bagian sebelumnya di namespace yang sama). Di paket aslinya
tiap bagian adalah modul terpisah dengan `from .modul_lain import ...`; di notebook ini import
antar-modul itu **sengaja dihapus** karena semua fungsi sudah berbagi satu namespace global — tidak
ada perubahan logika, hanya digabung fisik jadi satu file.

## ⚠️ Catatan penting sebelum dipakai untuk data skripsi sesungguhnya
1. **Redaksi PII tidak dipakai** — proposal yang diproses TIDAK dianonimkan. Nama, NIM, tanda tangan,
   dsb. akan ikut apa adanya ke `teks_unit` di output JSONL. Kalau perlindungan privasi tetap
   dibutuhkan, itu harus ditangani terpisah di luar notebook ini.
2. **Bagian B** — tabel kata kunci heading (`_KATA_KUNCI_UNIT`) adalah tebakan berdasar nama unit +
   contoh yang diberikan, BUKAN dikutip dari teks literal Pedoman v0.2. Kalibrasi ulang terhadap
   pedoman asli & sampel proposal riil sebelum dipakai untuk hasil skripsi.
3. **Bagian D** — `PETA_KOLOM_ANOTASI_DEFAULT`, `nama_sheet="Anotasi_Gabungan"`, dan
   `KODE_TAKSONOMI_AKTIF_DEFAULT` (21 kode) SUDAH dikonfirmasi dari workbook ground truth asli
   peneliti (per 2026-09-18) — bukan tebakan lagi. Tapi ini spesifik utk workbook ini; kalau
   struktur berubah/pakai workbook lain, tetap sesuaikan lewat `peta_kolom`/`nama_sheet`/
   `kode_taksonomi_valid`.
4. **Bagian C** — keputusan metodologis soal font "Title" 22pt vs Pasal 22 ayat (6) 16pt tetap di
   tangan peneliti; kode hanya melaporkan nilai apa adanya, tidak memutuskan mana yang benar.

Kode Python di bawah identik dengan yang ada di repo (`src/comply_proposal/preprocessing/*.py`),
hanya import antar-modul & blok `if __name__ == "__main__":` per-file yang dihapus (diganti satu
contoh pemanggilan end-to-end di bagian paling bawah notebook ini).


## Setup

Install dependensi (Colab sudah punya `pandas`; `python-docx`, `PyMuPDF`, `openpyxl` biasanya belum).

In [ ]:
!pip install -q "python-docx>=1.1.0" "PyMuPDF>=1.24.0" "openpyxl>=3.1.0"


Import seluruh dependensi eksternal yang dipakai di semua Bagian A–G (dipusatkan di sini, tidak diulang per bagian).

In [ ]:
from __future__ import annotations

import json
import random
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path
from typing import AbstractSet, Any, Dict, FrozenSet, List, Optional, Set, Tuple, Union

import pandas as pd

PathLike = Union[str, Path]


## Bagian A — Ekstraksi Teks (jalur JP/JC)

Modul ini bertanggung jawab HANYA untuk mengubah file .docx / .pdf menjadi
teks yang bersih dari artefak ekstraksi (header/footer berulang, nomor
halaman berulang, spasi ganda), TANPA menghilangkan informasi struktural
(nama gaya/heading, nomor bab/pasal, batas section/halaman) — informasi
struktural itu dibutuhkan oleh modul segmentasi unit (Bagian B).

Asumsi:
- PEMBARUAN: tahap redaksi PII terpisah TIDAK dipakai di implementasi ini
  (keputusan peneliti). Modul ini TIDAK melakukan deteksi/redaksi PII apa
  pun, dan data yang diproses TIDAK dianonimkan — nama, NIM, tanda tangan,
  dan identitas lain yang ada di dokumen asli akan ikut masuk APA ADANYA
  ke `teks_unit` pada output JSONL (Bagian G). Kalau perlindungan privasi
  tetap dibutuhkan (mis. sebelum data dibagikan ke pihak lain), itu harus
  ditangani terpisah di luar modul ini — JANGAN anggap pipeline ini sudah
  menanganinya.
- Deteksi placeholder redaksi (mis. "[DIREDAKSI]") di bawah tetap
  dipertahankan sebagai jaring pengaman murni (berjaga-jaga kalau ada sisa
  penanda manual di sebagian dokumen), TAPI karena tahap redaksi tidak
  lagi dipakai secara sistematis, jangan mengandalkan mekanisme ini
  sebagai bentuk perlindungan PII yang sebenarnya.
- Ekstraksi metadata tata letak (font, margin) BUKAN tanggung jawab modul
  ini — itu ada di modul terpisah untuk jalur JM (Bagian C), karena JM wajib
  membaca langsung dari file asli, bukan dari hasil ekstraksi teks di sini.
- Deteksi "kemungkinan hasil pindai/scan" pada modul ini hanya menghasilkan
  sinyal mentah (`kemungkinan_hasil_pindai`); keputusan akhir menandai
  proposal sebagai NA-01 diambil pada tahap penggabungan data (Bagian D),
  supaya satu proposal yang bermasalah tidak menghentikan seluruh batch.


In [ ]:
PathLike = Union[str, Path]


class EkstraksiError(Exception):
    """Dilempar untuk kegagalan ekstraksi yang terduga (file rusak/tidak bisa dibuka/kosong).

    Sengaja dipisahkan dari exception generik supaya pemanggil batch (lihat
    Bagian G) bisa menangkap ini secara spesifik, mencatatnya ke log, dan
    melanjutkan ke proposal berikutnya tanpa menghentikan seluruh proses.
    """


# ---------------------------------------------------------------------------
# Normalisasi teks & deteksi placeholder redaksi (dipakai bersama DOCX & PDF)
# ---------------------------------------------------------------------------

_POLA_SPASI_UNICODE = re.compile(
    "[\u00A0\u2000-\u200A\u202F\u3000\u200B]"
)
_POLA_SPASI_GANDA = re.compile(r"[ \t]{2,}")
_POLA_BARIS_KOSONG_BERLEBIH = re.compile(r"\n{3,}")
_TABEL_KUTIP_PINTAR = str.maketrans(
    {
        "‘": "'",
        "’": "'",
        "“": '"',
        "”": '"',
    }
)

# Token placeholder redaksi yang mungkin masih tersisa di teks. Daftar ini
# sengaja eksplisit (bukan pola "[...]" umum) supaya tidak salah menandai
# kurung siku yang memang bagian isi proposal (mis. kutipan referensi).
_POLA_PLACEHOLDER_REDAKSI = re.compile(
    r"\[\s*(DIREDAKSI|REDACTED|NAMA\s+DIREDAKSI|NIM\s+DIREDAKSI|TTD\s+DIREDAKSI)\s*\]",
    re.IGNORECASE,
)


def _normalisasi_teks(teks: str) -> str:
    """Merapikan spasi & tanda kutip tanpa mengubah struktur baris/paragraf.

    Langkah (sengaja terbatas & didokumentasikan, karena teks ini juga
    dipakai untuk pemeriksaan BHS/FMT — normalisasi berlebihan bisa
    mengaburkan pelanggaran yang sebenarnya):
    1. Samakan berbagai karakter spasi unicode (nbsp, dll.) jadi spasi biasa,
       TERMASUK zero-width space (U+200B) -- ditemukan empiris (2026-09-18)
       pada PDF hasil ekspor tool tertentu yang menyisipkan U+200B sebagai
       SATU-SATUNYA pemisah antar-kata (tanpa spasi biasa sama sekali, mis.
       "1.1​​Latar​​Belakang"). Karena U+200B bukan
       whitespace menurut `\s` regex Python, ini bikin SEMUA pola heading
       Bagian B/C gagal total pada dokumen yang terkena (bukan cuma fallback
       ke pola lebih lemah -- headingnya sungguh tidak ketemu sama sekali).
    2. Ratakan spasi/tab ganda dalam satu baris jadi satu spasi.
    3. Samakan tanda kutip pintar jadi tanda kutip lurus.
    4. Batasi baris kosong berturut-turut maksimal satu (antar paragraf).

    Input kosong/None-safe: mengembalikan string kosong.
    """
    if not teks:
        return ""

    teks = _POLA_SPASI_UNICODE.sub(" ", teks)
    teks = teks.translate(_TABEL_KUTIP_PINTAR)
    baris_dirapikan = [_POLA_SPASI_GANDA.sub(" ", b).strip() for b in teks.split("\n")]
    teks = "\n".join(baris_dirapikan)
    teks = _POLA_BARIS_KOSONG_BERLEBIH.sub("\n\n", teks)
    return teks.strip()


def _mengandung_placeholder_redaksi(teks: str) -> bool:
    """True jika teks masih mengandung sisa placeholder redaksi PII."""
    return bool(_POLA_PLACEHOLDER_REDAKSI.search(teks or ""))


# ---------------------------------------------------------------------------
# Bagian DOCX
# ---------------------------------------------------------------------------


def _petakan_indeks_section_docx(dokumen: Any) -> List[int]:
    """Memetakan setiap paragraf level-dokumen ke indeks section (0-based).

    python-docx tidak menyediakan pemetaan ini secara langsung. Section baru
    ditandai oleh elemen <w:sectPr> yang bersarang di <w:pPr> paragraf
    TERAKHIR pada section tsb (section terakhir dalam dokumen justru
    sectPr-nya anak langsung <w:body>, bukan di dalam paragraf, sehingga
    tidak menambah indeks lagi setelah paragraf terakhir).

    Pemetaan ini penting karena template resmi punya margin berbeda per
    section (Sampul vs badan proposal vs lampiran) — dibutuhkan oleh jalur
    JM (Bagian C), meski modul ekstraksi teks ini sendiri tidak membaca
    margin.

    Urutan hasil mengikuti urutan `dokumen.paragraphs` (paragraf di dalam
    tabel tidak termasuk, sama seperti behavior `document.paragraphs`).
    """
    from docx.oxml.ns import qn

    indeks_section = 0
    hasil: List[int] = []
    body = dokumen.element.body
    for anak in body.iterchildren():
        if anak.tag == qn("w:p"):
            hasil.append(indeks_section)
            ppr = anak.find(qn("w:pPr"))
            if ppr is not None and ppr.find(qn("w:sectPr")) is not None:
                indeks_section += 1
    return hasil


def ekstrak_docx(path: PathLike) -> Dict[str, Any]:
    """Mengekstrak teks terstruktur dari file .docx.

    Input:
        path: path ke file .docx. PII di dalam dokumen TIDAK diredaksi oleh
            modul ini maupun tahap sebelumnya (lihat catatan asumsi di
            docstring modul) — akan ikut apa adanya di `teks_penuh`/`unit_teks`.

    Output (dict):
        {
            "path_asal": str,
            "format_asli": "docx",
            "teks_penuh": str,              # gabungan semua paragraf non-kosong
            "unit_teks": [                  # satu entri per paragraf level-dokumen
                {
                    "indeks_paragraf": int,
                    "indeks_section": int | None,   # lihat _petakan_indeks_section_docx
                    "gaya": str | None,              # nama style, mis. "Heading 1"
                    "level_outline": int | None,
                    "teks": str,                     # sudah dinormalisasi
                    "mengandung_placeholder_redaksi": bool,
                },
                ...
            ],
            "jumlah_section": int,
            "peringatan": [str, ...],
        }

    Header/footer TIDAK perlu dibersihkan secara heuristik di sini seperti
    pada PDF: pada format .docx, header/footer tersimpan sebagai objek
    terpisah per section (section.header / section.footer), bukan
    tercampur ke dalam alur paragraf body, sehingga `dokumen.paragraphs`
    sudah otomatis tidak menyertakannya.

    Raises:
        EkstraksiError: jika file tidak bisa dibuka/dibaca sebagai .docx.
    """
    try:
        from docx import Document
    except ImportError as exc:  # pragma: no cover
        raise EkstraksiError(
            "Paket 'python-docx' belum terpasang. Jalankan: pip install python-docx"
        ) from exc

    path = Path(path)
    peringatan: List[str] = []

    try:
        dokumen = Document(str(path))
    except Exception as exc:
        raise EkstraksiError(f"Gagal membuka file DOCX '{path.name}': {exc}") from exc

    indeks_section_per_paragraf = _petakan_indeks_section_docx(dokumen)

    unit_teks: List[Dict[str, Any]] = []
    potongan_teks_penuh: List[str] = []

    for i, paragraf in enumerate(dokumen.paragraphs):
        teks_mentah = paragraf.text
        teks_bersih = _normalisasi_teks(teks_mentah)

        gaya = None
        try:
            if paragraf.style is not None:
                gaya = paragraf.style.name
        except Exception:
            gaya = None

        level_outline = None
        try:
            level_outline = paragraf.paragraph_format.outline_level
        except Exception:
            level_outline = None

        unit_teks.append(
            {
                "indeks_paragraf": i,
                "indeks_section": (
                    indeks_section_per_paragraf[i]
                    if i < len(indeks_section_per_paragraf)
                    else None
                ),
                "gaya": gaya,
                "level_outline": level_outline,
                "teks": teks_bersih,
                "mengandung_placeholder_redaksi": _mengandung_placeholder_redaksi(teks_mentah),
            }
        )
        if teks_bersih:
            potongan_teks_penuh.append(teks_bersih)

    if not potongan_teks_penuh:
        peringatan.append(
            "Tidak ada teks yang berhasil diekstrak dari file DOCX ini (dokumen mungkin kosong)."
        )

    return {
        "path_asal": str(path),
        "format_asli": "docx",
        "teks_penuh": "\n".join(potongan_teks_penuh),
        "unit_teks": unit_teks,
        "jumlah_section": len(dokumen.sections),
        "peringatan": peringatan,
    }


# ---------------------------------------------------------------------------
# Bagian PDF
# ---------------------------------------------------------------------------

_RASIO_ZONA_HEADER_FOOTER = 0.12
_AMBANG_PROPORSI_HALAMAN_UNTUK_ARTEFAK = 0.5
_MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK = 3
_AMBANG_KARAKTER_PER_HALAMAN_UNTUK_PINDAI = 20.0

_POLA_NOMOR_HALAMAN = re.compile(
    r"^(halaman|page|hal\.?)?\s*\.?\s*\d{1,4}(\s*(dari|of|/)\s*\d{1,4})?\s*\.?\s*$",
    re.IGNORECASE,
)
# Validasi angka romawi kanonik (bukan sekadar cek karakter), untuk mengurangi
# false positive terhadap kata singkat berbahasa Indonesia (mis. "di").
_POLA_ANGKA_ROMAWI = re.compile(
    r"^M{0,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$",
    re.IGNORECASE,
)


def _adalah_kandidat_nomor_halaman(teks: str) -> bool:
    """True jika satu baris teks terlihat seperti nomor halaman berdiri sendiri.

    Dipakai HANYA pada baris yang sudah dipastikan berada di zona
    header/footer (lihat pemanggil) dan HANYA dianggap artefak jika pola ini
    konsisten muncul di posisi yang sama pada banyak halaman (lihat
    `_deteksi_pola_nomor_halaman_berulang`) — supaya kata pendek yang
    kebetulan mirip angka romawi (mis. "di") tidak salah terhapus hanya
    karena muncul sekali di zona footer suatu halaman.
    """
    t = teks.strip()
    if not t:
        return False
    if _POLA_NOMOR_HALAMAN.match(t):
        return True
    if 1 < len(t) <= 6 and _POLA_ANGKA_ROMAWI.match(t):
        return True
    return False


def _ambil_baris_halaman(halaman: Any) -> List[Dict[str, Any]]:
    """Mengambil baris teks + posisi (bbox) dari satu halaman PDF via PyMuPDF.

    Diurutkan dari atas ke bawah lalu kiri ke kanan, dengan asumsi tata letak
    satu kolom (sesuai template proposal resmi). Baris kosong dibuang.
    """
    data = halaman.get_text("dict")
    baris_list: List[Dict[str, Any]] = []
    for blok in data.get("blocks", []):
        for baris in blok.get("lines", []):
            teks_baris = "".join(span.get("text", "") for span in baris.get("spans", [])).strip()
            if not teks_baris:
                continue
            x0, y0, x1, y1 = baris.get("bbox", (0.0, 0.0, 0.0, 0.0))
            baris_list.append({"teks": teks_baris, "x0": x0, "y0": y0, "x1": x1, "y1": y1})

    baris_list.sort(key=lambda b: (round(b["y0"], 1), b["x0"]))
    return baris_list


def _deteksi_artefak_header_footer(
    baris_per_halaman: List[List[Dict[str, Any]]],
    tinggi_halaman: List[float],
) -> Set[str]:
    """Mengidentifikasi teks yang tampil PERSIS SAMA di zona header/footer
    (12% teratas / 12% terbawah halaman) pada banyak halaman -> dianggap
    artefak header/footer berulang (mis. judul dokumen, nama institusi),
    bukan bagian isi proposal.

    Perbandingan dilakukan setelah normalisasi & lowercasing agar variasi
    spasi tidak mengelabui deteksi. Butuh minimal
    `_MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK` halaman supaya "berulang" punya
    arti statistik; dokumen yang lebih pendek dari itu tidak melalui langkah
    ini (dianggap terlalu berisiko salah tandai).

    Return: himpunan teks (sudah dinormalisasi, huruf kecil) yang dianggap
    artefak header/footer.
    """
    jumlah_halaman = len(baris_per_halaman)
    if jumlah_halaman < _MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK:
        return set()

    kemunculan: Dict[str, Set[int]] = defaultdict(set)
    for idx_halaman, (baris_list, tinggi) in enumerate(zip(baris_per_halaman, tinggi_halaman)):
        zona = tinggi * _RASIO_ZONA_HEADER_FOOTER
        for baris in baris_list:
            di_header = baris["y1"] <= zona
            di_footer = baris["y0"] >= (tinggi - zona)
            if not (di_header or di_footer):
                continue
            kunci = _normalisasi_teks(baris["teks"]).lower()
            if kunci:
                kemunculan[kunci].add(idx_halaman)

    ambang = max(
        _MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK,
        int(jumlah_halaman * _AMBANG_PROPORSI_HALAMAN_UNTUK_ARTEFAK),
    )
    return {kunci for kunci, halaman_muncul in kemunculan.items() if len(halaman_muncul) >= ambang}


def _deteksi_pola_nomor_halaman_berulang(
    baris_per_halaman: List[List[Dict[str, Any]]],
    tinggi_halaman: List[float],
) -> Dict[str, bool]:
    """Mendeteksi apakah zona header dan/atau footer SECARA KONSISTEN berisi
    nomor halaman (nilainya berubah tiap halaman, sehingga tidak tertangkap
    oleh `_deteksi_artefak_header_footer` yang membandingkan teks persis
    sama). Yang dibandingkan di sini adalah POLA (mis. "12", "Halaman 12
    dari 45"), bukan nilainya.

    Return: {"header": bool, "footer": bool} — True jika zona tsb dianggap
    berisi penomoran halaman berulang pada mayoritas halaman.
    """
    jumlah_halaman = len(baris_per_halaman)
    if jumlah_halaman < _MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK:
        return {"header": False, "footer": False}

    halaman_dengan_nomor_di_header: Set[int] = set()
    halaman_dengan_nomor_di_footer: Set[int] = set()

    for idx_halaman, (baris_list, tinggi) in enumerate(zip(baris_per_halaman, tinggi_halaman)):
        zona = tinggi * _RASIO_ZONA_HEADER_FOOTER
        for baris in baris_list:
            if not _adalah_kandidat_nomor_halaman(baris["teks"]):
                continue
            if baris["y1"] <= zona:
                halaman_dengan_nomor_di_header.add(idx_halaman)
            elif baris["y0"] >= (tinggi - zona):
                halaman_dengan_nomor_di_footer.add(idx_halaman)

    ambang = max(
        _MINIMUM_HALAMAN_UNTUK_DETEKSI_ARTEFAK,
        int(jumlah_halaman * _AMBANG_PROPORSI_HALAMAN_UNTUK_ARTEFAK),
    )
    return {
        "header": len(halaman_dengan_nomor_di_header) >= ambang,
        "footer": len(halaman_dengan_nomor_di_footer) >= ambang,
    }


def ekstrak_pdf(path: PathLike) -> Dict[str, Any]:
    """Mengekstrak teks terstruktur dari file .pdf.

    Input:
        path: path ke file .pdf. PII di dalam dokumen TIDAK diredaksi oleh
            modul ini maupun tahap sebelumnya (lihat catatan asumsi di
            docstring modul) — akan ikut apa adanya di `teks_penuh`/`unit_teks`.

    Output (dict):
        {
            "path_asal": str,
            "format_asli": "pdf",
            "teks_penuh": str,              # gabungan semua halaman non-kosong
            "unit_teks": [                  # satu entri per halaman
                {
                    "indeks_halaman": int,
                    "teks": str,             # setelah dibersihkan header/footer & dinormalisasi
                    "jumlah_baris_terhapus_sebagai_artefak": int,
                    "mengandung_placeholder_redaksi": bool,
                },
                ...
            ],
            "jumlah_halaman": int,
            "kemungkinan_hasil_pindai": bool,   # sinyal mentah utk kandidat NA-01
            "header_footer_terdeteksi": [str, ...],  # utk transparansi/audit
            "peringatan": [str, ...],
        }

    Catatan: fungsi ini TIDAK memutuskan status NA-01 secara final — itu
    tanggung jawab modul penggabungan data (Bagian D), yang bisa
    mempertimbangkan konteks lain (mis. ukuran file, jumlah gambar).
    Fungsi ini hanya melaporkan sinyal `kemungkinan_hasil_pindai`.

    Raises:
        EkstraksiError: jika file tidak bisa dibuka atau tidak punya halaman.
    """
    try:
        import pymupdf  # PyMuPDF (nama modul 'pymupdf'; 'fitz' adalah alias lama/deprecated)
    except ImportError as exc:  # pragma: no cover
        raise EkstraksiError(
            "Paket 'PyMuPDF' belum terpasang. Jalankan: pip install PyMuPDF"
        ) from exc

    path = Path(path)
    peringatan: List[str] = []

    try:
        dokumen = pymupdf.open(str(path))
    except Exception as exc:
        raise EkstraksiError(f"Gagal membuka file PDF '{path.name}': {exc}") from exc

    try:
        jumlah_halaman = dokumen.page_count
        if jumlah_halaman == 0:
            raise EkstraksiError(f"File PDF '{path.name}' tidak memiliki halaman.")

        baris_per_halaman: List[List[Dict[str, Any]]] = []
        tinggi_halaman: List[float] = []
        for halaman in dokumen:
            tinggi_halaman.append(float(halaman.rect.height))
            baris_per_halaman.append(_ambil_baris_halaman(halaman))

        artefak_teks_persis = _deteksi_artefak_header_footer(baris_per_halaman, tinggi_halaman)
        pola_nomor_berulang = _deteksi_pola_nomor_halaman_berulang(
            baris_per_halaman, tinggi_halaman
        )

        unit_teks: List[Dict[str, Any]] = []
        potongan_teks_penuh: List[str] = []
        total_karakter_mentah = 0

        for idx_halaman, (baris_list, tinggi) in enumerate(zip(baris_per_halaman, tinggi_halaman)):
            zona = tinggi * _RASIO_ZONA_HEADER_FOOTER
            baris_dipertahankan: List[str] = []
            jumlah_terhapus = 0

            for baris in baris_list:
                total_karakter_mentah += len(baris["teks"])
                di_header = baris["y1"] <= zona
                di_footer = baris["y0"] >= (tinggi - zona)
                kunci = _normalisasi_teks(baris["teks"]).lower()

                if kunci in artefak_teks_persis:
                    jumlah_terhapus += 1
                    continue
                if di_header and pola_nomor_berulang["header"] and _adalah_kandidat_nomor_halaman(
                    baris["teks"]
                ):
                    jumlah_terhapus += 1
                    continue
                if di_footer and pola_nomor_berulang["footer"] and _adalah_kandidat_nomor_halaman(
                    baris["teks"]
                ):
                    jumlah_terhapus += 1
                    continue

                baris_dipertahankan.append(baris["teks"])

            teks_halaman = _normalisasi_teks("\n".join(baris_dipertahankan))
            mengandung_placeholder = any(
                _mengandung_placeholder_redaksi(b) for b in baris_dipertahankan
            )

            unit_teks.append(
                {
                    "indeks_halaman": idx_halaman,
                    "teks": teks_halaman,
                    "jumlah_baris_terhapus_sebagai_artefak": jumlah_terhapus,
                    "mengandung_placeholder_redaksi": mengandung_placeholder,
                }
            )
            if teks_halaman:
                potongan_teks_penuh.append(teks_halaman)

        rata_rata_karakter_per_halaman = total_karakter_mentah / jumlah_halaman
        kemungkinan_hasil_pindai = (
            rata_rata_karakter_per_halaman < _AMBANG_KARAKTER_PER_HALAMAN_UNTUK_PINDAI
        )

        if kemungkinan_hasil_pindai:
            peringatan.append(
                "Rata-rata karakter per halaman sangat rendah "
                f"({rata_rata_karakter_per_halaman:.1f}); kemungkinan file ini hasil "
                "pindai/scan (kandidat NA-01). Keputusan akhir NA-01 diambil pada "
                "tahap penggabungan data (Bagian D)."
            )
        if not potongan_teks_penuh:
            peringatan.append("Tidak ada teks yang berhasil diekstrak dari file PDF ini.")

        return {
            "path_asal": str(path),
            "format_asli": "pdf",
            "teks_penuh": "\n\n".join(potongan_teks_penuh),
            "unit_teks": unit_teks,
            "jumlah_halaman": jumlah_halaman,
            "kemungkinan_hasil_pindai": kemungkinan_hasil_pindai,
            "header_footer_terdeteksi": sorted(artefak_teks_persis),
            "peringatan": peringatan,
        }
    finally:
        dokumen.close()


# ---------------------------------------------------------------------------
# Fungsi pemilih format (dipakai modul-modul selanjutnya, mis. Bagian D/G)
# ---------------------------------------------------------------------------


def ekstrak_dokumen(path: PathLike) -> Dict[str, Any]:
    """Mendeteksi format dari ekstensi file lalu memanggil ekstraktor yang sesuai.

    Raises:
        EkstraksiError: jika ekstensi file bukan .docx atau .pdf, atau jika
            ekstraksi gagal (lihat `ekstrak_docx`/`ekstrak_pdf`).
    """
    path = Path(path)
    ekstensi = path.suffix.lower()
    if ekstensi == ".docx":
        return ekstrak_docx(path)
    if ekstensi == ".pdf":
        return ekstrak_pdf(path)
    raise EkstraksiError(
        f"Format file tidak didukung untuk '{path.name}' (ekstensi '{ekstensi}'). "
        "Hanya .docx dan .pdf yang didukung."
    )


## Bagian B — Segmentasi ke 16 Unit Anotasi (U00–U15)

Modul ini mengambil hasil ekstraksi teks dari Bagian A (`ekstraksi_teks.py`)
dan memecahnya menjadi 16 unit sesuai Panduan Anotasi dan Taksonomi
Pelanggaran v0.2, berdasarkan deteksi heading/subheading di dalam teks.

PENTING — batasan sumber kebenaran:
Saya (asisten) TIDAK punya akses ke teks literal Pedoman Penulisan Proposal
Kegiatan Mahasiswa v0.2, hanya ke daftar nama 16 unit dan dua contoh heading
("1.1 Latar Belakang" -> U03, "2.5 Struktur Kepanitiaan" -> U10) yang
diberikan di percakapan. Tabel kata kunci `_KATA_KUNCI_UNIT` di bawah adalah
tebakan terbaik berdasarkan nama unit + contoh tsb, BUKAN dikutip langsung
dari Pedoman v0.2. Peneliti WAJIB mengkalibrasi ulang tabel ini terhadap
teks asli Pedoman v0.2 dan sampel proposal riil sebelum dipakai untuk hasil
skripsi — jangan anggap tabel ini sudah final.

Asumsi:
- Input `teks_terstruktur` adalah dict keluaran `ekstrak_docx`/`ekstrak_pdf`
  (Bagian A), BUKAN path file mentah.
- U00 (Dokumen) tidak punya heading tekstual — propertinya (bahasa, kertas,
  margin, huruf, penomoran bab) diisi lewat metadata JM (Bagian C), sehingga
  di modul ini U00 selalu dicatat dengan teks kosong + metode "tidak_berlaku".
- U01 (Sampul) biasanya tidak punya heading eksplisit di badan dokumen;
  secara default seluruh konten SEBELUM heading unit pertama yang terdeteksi
  dianggap sebagai U01, kecuali ditemukan heading eksplisit untuknya.
- Unit yang sama sekali tidak terdeteksi TETAP dicatat (teks kosong,
  ditemukan=False) — bukan dihilangkan dari output — karena dibutuhkan oleh
  jalur JC (checklist keberadaan).
- Deteksi heading yang tidak lolos pola baku (nomor + kata kunci) dicatat
  sebagai fallback dengan `metode_deteksi` & `catatan` yang eksplisit,
  supaya tidak gagal diam-diam.


In [ ]:
# ---------------------------------------------------------------------------
# Daftar kanonik 16 unit anotasi
# ---------------------------------------------------------------------------

DAFTAR_UNIT: List[Tuple[str, str]] = [
    ("U00", "Dokumen"),
    ("U01", "Sampul"),
    ("U02", "Daftar Isi"),
    ("U03", "Latar Belakang"),
    ("U04", "Maksud/Tujuan/Sasaran"),
    ("U05", "Indikator Keberhasilan"),
    ("U06", "Nama Kegiatan"),
    ("U07", "Bentuk Kegiatan"),
    ("U08", "Waktu dan Tempat"),
    ("U09", "Peserta"),
    ("U10", "Struktur Kepanitiaan"),
    ("U11", "Susunan Acara"),
    ("U12", "Perencanaan Keuangan"),
    ("U13", "Penutup dan Pengesahan"),
    ("U14", "Lampiran I (Formulir Manajemen Risiko)"),
    ("U15", "Lampiran II (Timeline)"),
]

ID_UNIT_VALID: Set[str] = {uid for uid, _ in DAFTAR_UNIT}


# ---------------------------------------------------------------------------
# Pola heading per unit (lihat catatan kalibrasi di docstring modul)
# ---------------------------------------------------------------------------

# Awalan penomoran umum: "1.1 ", "2.5. ", "BAB III ", "iv. ", dll.
_POLA_AWALAN_NOMOR = r"^\s*(bab\s+)?(\d+(\.\d+){0,3}|[ivxlcdm]+)[.\)]?\s+"

_KATA_KUNCI_UNIT: Dict[str, List[str]] = {
    "U01": [r"sampul"],
    "U02": [r"daftar\s+isi"],
    "U03": [r"latar\s+belakang"],
    "U04": [
        r"maksud.{0,20}tujuan",
        r"tujuan.{0,20}sasaran",
        r"maksud\s+dan\s+tujuan",
    ],
    "U05": [r"indikator\s+keberhasilan"],
    "U06": [r"nama\s+kegiatan"],
    "U07": [r"bentuk\s+(dan\s+jenis\s+)?kegiatan"],
    "U08": [r"waktu\s+dan\s+tempat", r"waktu.{0,20}tempat\s+pelaksanaan"],
    "U09": [r"peserta\s+kegiatan", r"peserta"],
    "U10": [r"struktur\s+kepanitiaan", r"susunan\s+kepanitiaan", r"susunan\s+panitia"],
    "U11": [r"susunan\s+acara", r"rundown\s+acara", r"jadwal\s+acara"],
    "U12": [
        r"perencanaan\s+keuangan",
        r"rencana\s+anggaran",
        r"anggaran\s+(biaya|dana)",
        r"rincian\s+anggaran",
    ],
    "U13": [r"penutup", r"pengesahan"],
    "U14": [
        r"lampiran\s+(i|1)(?!\w)",
        r"formulir\s+manajemen\s+risiko",
        r"manajemen\s+risiko",
    ],
    "U15": [
        r"lampiran\s+(ii|2)(?!\w)",
        r"timeline",
        r"jadwal\s+kegiatan\s*\(?\s*timeline\s*\)?",
    ],
}

_POLA_KETAT: Dict[str, re.Pattern] = {
    uid: re.compile(_POLA_AWALAN_NOMOR + "(" + "|".join(kk) + ")", re.IGNORECASE)
    for uid, kk in _KATA_KUNCI_UNIT.items()
}
_POLA_LONGGAR: Dict[str, re.Pattern] = {
    uid: re.compile("(" + "|".join(kk) + ")", re.IGNORECASE) for uid, kk in _KATA_KUNCI_UNIT.items()
}

_PANJANG_MAKS_BARIS_FALLBACK = 120
_AKHIRAN_BUKAN_HEADING = (".", ",", ";")
_AWALAN_GAYA_HEADING = ("heading", "judul")  # "Judul" = lokalisasi Indonesia utk style Word

# Baris entri Daftar Isi (mis. "1.1 Latar Belakang ................... 3") secara
# tekstual cocok PERSIS dengan pola heading unit yang sebenarnya -- tanpa
# pengecualian ini, entri Daftar Isi akan "membajak" anchor unit tsb (karena
# anchor pertama yang menang) sebelum heading asli di badan dokumen tercapai,
# membuat teks unit itu berisi baris Daftar Isi, BUKAN paragraf sungguhan.
# Ditemukan empiris (2026-09-19) pada ~29% proposal riil yang diuji peneliti.
# Titik-titik penuntun (dot leader) sepanjang ini praktis tidak pernah muncul
# di prosa biasa, jadi dipakai sebagai penanda "ini entri Daftar Isi, bukan
# heading" -- baris yang cocok langsung dianggap BUKAN kandidat heading sama
# sekali, apa pun isinya.
_POLA_ENTRI_DAFTAR_ISI = re.compile(r"\.{4,}")


def _cari_unit_untuk_baris(teks_baris: str, gaya: Optional[str]) -> Optional[Tuple[str, str]]:
    """Mencoba mencocokkan satu baris/paragraf dengan pola heading salah satu unit.

    Prioritas pencocokan (berhenti di percobaan pertama yang berhasil):
    0. Baris berpola entri Daftar Isi (titik penuntun panjang) -> SELALU None,
       tidak pernah dianggap heading (lihat catatan `_POLA_ENTRI_DAFTAR_ISI`).
    1. Pola ketat: awalan nomor/bab + kata kunci -> metode "heading_bernomor".
    2. Kalau gaya paragraf menandakan heading Word (Heading */Judul *) ->
       coba pola longgar (tanpa syarat nomor) -> metode "heading_gaya".
    3. Pola longgar dengan batas panjang baris & tidak diakhiri tanda baca
       kalimat -> metode "heading_kata_kunci_fallback".

    Return None jika tidak ada yang cocok. Baris kosong selalu None.
    """
    baris = teks_baris.strip()
    if not baris:
        return None
    if _POLA_ENTRI_DAFTAR_ISI.search(baris):
        return None

    for unit_id in _KATA_KUNCI_UNIT:
        if _POLA_KETAT[unit_id].match(baris):
            return unit_id, "heading_bernomor"

    gaya_menandakan_heading = bool(gaya) and gaya.lower().startswith(_AWALAN_GAYA_HEADING)
    if gaya_menandakan_heading:
        for unit_id in _KATA_KUNCI_UNIT:
            if _POLA_LONGGAR[unit_id].match(baris):
                return unit_id, "heading_gaya"

    if len(baris) <= _PANJANG_MAKS_BARIS_FALLBACK and not baris.endswith(_AKHIRAN_BUKAN_HEADING):
        for unit_id in _KATA_KUNCI_UNIT:
            if _POLA_LONGGAR[unit_id].match(baris):
                return unit_id, "heading_kata_kunci_fallback"

    return None


# ---------------------------------------------------------------------------
# Adaptasi struktur hasil Bagian A (beda antara docx & pdf) ke representasi seragam
# ---------------------------------------------------------------------------


def _ambil_blok_berurutan(teks_terstruktur: Dict[str, Any], format_asli: str) -> List[Dict[str, Any]]:
    """Menyeragamkan keluaran Bagian A menjadi daftar blok teks berurutan.

    - format_asli == "docx": satu blok = satu paragraf (`gaya` = nama style
      paragraf tsb, dipakai sebagai sinyal tambahan deteksi heading).
    - format_asli == "pdf": satu blok = satu baris teks dalam satu halaman
      (`gaya` selalu None; info style tidak tersedia dari ekstraksi PDF —
      itu ranah modul JM/Bagian C, bukan modul ini).

    Urutan blok mengikuti urutan alami dokumen sehingga potongan teks antar
    dua anchor heading bisa diambil dengan slicing sederhana.
    """
    format_asli = (format_asli or "").lower()
    blok_list: List[Dict[str, Any]] = []

    if format_asli == "docx":
        for unit in teks_terstruktur.get("unit_teks", []):
            blok_list.append({"teks": unit.get("teks", ""), "gaya": unit.get("gaya")})
    elif format_asli == "pdf":
        for halaman in teks_terstruktur.get("unit_teks", []):
            for baris in halaman.get("teks", "").split("\n"):
                blok_list.append({"teks": baris, "gaya": None})
    else:
        raise ValueError(f"format_asli tidak dikenal: '{format_asli}'. Gunakan 'docx' atau 'pdf'.")

    return blok_list


def _normalisasi_gabungan(potongan: List[str]) -> str:
    """Menggabungkan beberapa potongan teks (baris/paragraf) jadi satu teks unit."""
    gabungan = "\n".join(p for p in potongan if p)
    return _normalisasi_teks(gabungan)


# ---------------------------------------------------------------------------
# Fungsi utama
# ---------------------------------------------------------------------------


def segmentasi_unit(teks_terstruktur: Dict[str, Any], format_asli: str) -> Dict[str, Any]:
    """Memecah teks hasil ekstraksi (Bagian A) menjadi 16 unit anotasi.

    Input:
        teks_terstruktur: dict keluaran `ekstrak_docx`/`ekstrak_pdf`/`ekstrak_dokumen`.
        format_asli: "docx" atau "pdf" (biasanya sama dengan
            `teks_terstruktur["format_asli"]`, dipisah sebagai parameter agar
            fungsi ini tidak diam-diam bergantung pada field internal Bagian A).

    Output (dict):
        {
            "unit": {
                "U00": {"teks": "", "ditemukan": False, "metode_deteksi": "tidak_berlaku", "catatan": "..."},
                "U01": {"teks": str, "ditemukan": bool, "metode_deteksi": str, "catatan": str | None},
                ...
                "U15": {...},
            },
            "peringatan": [str, ...],   # peringatan level dokumen (unit hilang, gagal total, dst.)
        }

    `metode_deteksi` salah satu dari:
        "tidak_berlaku"                  -> khusus U00
        "tidak_ditemukan"                -> unit tidak ada sama sekali di dokumen
        "posisi_awal_dokumen"            -> khusus U01 tanpa heading eksplisit
        "heading_bernomor"               -> pola ketat (nomor + kata kunci), paling andal
        "heading_gaya"                   -> gaya Word Heading/Judul + kata kunci, tanpa nomor
        "heading_kata_kunci_fallback"    -> kata kunci saja, tanpa nomor/gaya (paling lemah)
        "gagal_segmentasi_seluruh_dokumen" -> tidak ada heading apa pun terdeteksi

    Raises:
        ValueError: jika `format_asli` bukan "docx"/"pdf".
    """
    blok_list = _ambil_blok_berurutan(teks_terstruktur, format_asli)

    anchor: List[Tuple[int, str, str]] = []
    unit_sudah_anchor: Set[str] = set()
    for i, blok in enumerate(blok_list):
        hasil = _cari_unit_untuk_baris(blok["teks"], blok.get("gaya"))
        if hasil is None:
            continue
        unit_id, metode = hasil
        if unit_id in unit_sudah_anchor:
            continue
        anchor.append((i, unit_id, metode))
        unit_sudah_anchor.add(unit_id)
    anchor.sort(key=lambda a: a[0])

    unit_hasil: Dict[str, Dict[str, Any]] = {
        uid: {"teks": "", "ditemukan": False, "metode_deteksi": "tidak_ditemukan", "catatan": None}
        for uid, _ in DAFTAR_UNIT
    }
    unit_hasil["U00"] = {
        "teks": "",
        "ditemukan": False,
        "metode_deteksi": "tidak_berlaku",
        "catatan": (
            "U00 adalah properti tata letak keseluruhan dokumen (bahasa, kertas, "
            "margin, huruf, penomoran bab); diisi lewat metadata JM (Bagian C), "
            "bukan lewat segmentasi teks ini."
        ),
    }

    peringatan: List[str] = []

    if not anchor:
        peringatan.append(
            "Tidak ada satu pun heading unit yang terdeteksi di seluruh dokumen "
            "(kemungkinan format penomoran sangat menyimpang dari pola yang dikenali "
            "atau tabel kata kunci perlu dikalibrasi ulang). Seluruh isi disimpan "
            "sebagai U01 secara default; WAJIB ditinjau manual."
        )
        teks_gabungan = _normalisasi_gabungan([b["teks"] for b in blok_list])
        unit_hasil["U01"] = {
            "teks": teks_gabungan,
            "ditemukan": bool(teks_gabungan),
            "metode_deteksi": "gagal_segmentasi_seluruh_dokumen",
            "catatan": "Tidak ada heading terdeteksi; seluruh teks dokumen dianggap satu blok.",
        }
    else:
        indeks_anchor_pertama, unit_id_pertama, _ = anchor[0]
        if unit_id_pertama != "U01":
            teks_sampul = _normalisasi_gabungan([b["teks"] for b in blok_list[:indeks_anchor_pertama]])
            if teks_sampul:
                unit_hasil["U01"] = {
                    "teks": teks_sampul,
                    "ditemukan": True,
                    "metode_deteksi": "posisi_awal_dokumen",
                    "catatan": (
                        "U01 tidak punya heading eksplisit; diasumsikan semua konten "
                        f"sebelum heading unit pertama yang terdeteksi ({unit_id_pertama})."
                    ),
                }

        for idx_anchor, (indeks_blok, unit_id, metode) in enumerate(anchor):
            akhir = anchor[idx_anchor + 1][0] if idx_anchor + 1 < len(anchor) else len(blok_list)
            teks_unit = _normalisasi_gabungan([b["teks"] for b in blok_list[indeks_blok:akhir]])

            catatan = None
            if metode == "heading_kata_kunci_fallback":
                catatan = (
                    f"Heading '{unit_id}' terdeteksi lewat pencocokan kata kunci tanpa pola "
                    "penomoran baku (mis. mahasiswa mengubah format penomoran); perlu verifikasi manual."
                )
            elif metode == "heading_gaya":
                catatan = (
                    f"Heading '{unit_id}' terdeteksi lewat gaya paragraf Word (Heading/Judul) "
                    "tanpa pola kata kunci+nomor standar; perlu verifikasi manual."
                )

            unit_hasil[unit_id] = {
                "teks": teks_unit,
                "ditemukan": bool(teks_unit),
                "metode_deteksi": metode,
                "catatan": catatan,
            }
            if catatan:
                peringatan.append(f"[{unit_id}] {catatan}")

    unit_tidak_ditemukan = [
        uid for uid, info in unit_hasil.items() if uid != "U00" and not info["ditemukan"]
    ]
    if unit_tidak_ditemukan:
        peringatan.append(
            "Unit tidak ditemukan di dokumen ini (dicatat dengan teks kosong untuk "
            "jalur JC/checklist): " + ", ".join(sorted(unit_tidak_ditemukan))
        )

    return {"unit": unit_hasil, "peringatan": peringatan}


## Bagian C — Metadata Tata Letak (jalur JM)

PRINSIP UTAMA (WAJIB dijaga di seluruh modul ini): jalur JM tidak boleh
membaca margin/font dari teks hasil ekstraksi Bagian A. Semua fungsi di sini
membuka file asli sendiri (python-docx untuk .docx, PyMuPDF untuk .pdf) dan
sama sekali tidak menerima/menggunakan output `ekstrak_docx`/`ekstrak_pdf`
sebagai input. Satu-satunya helper yang diimpor dari Bagian A adalah
`_petakan_indeks_section_docx`, karena fungsi itu murni menelusuri struktur
XML dokumen (bukan teks hasil ekstraksi) untuk memetakan section — bug-prone
kalau diduplikasi, jadi sengaja dipakai bersama.

Temuan penting dari template resmi (WAJIB diperhatikan saat memakai output
modul ini, lihat juga konteks percakapan):
- Template resmi punya 3 section dengan margin BERBEDA (Sampul / badan
  proposal Bab I-III / Lampiran I-II). Karena itu setiap entri di sini
  SELALU menyertakan section/halaman mana yang diukur — JANGAN mengambil
  satu margin lalu menganggapnya berlaku untuk seluruh dokumen.
- Style bawaan "Title" pada file template resmi = 22pt, sedangkan Pedoman
  v0.2 Pasal 22 ayat (6) huruf a menyebut 16pt untuk identitas sampul.
  Modul ini TIDAK memutuskan mana yang "benar" — ia hanya melaporkan apa
  adanya nilai yang berhasil diresolusi (lihat `sumber_ukuran_font`, akan
  bernilai "gaya:Title" untuk kasus ini). Keputusan metodologis (pakai
  16pt sesuai ketentuan tertulis, atau 22pt sesuai file template) ada di
  tangan peneliti, bukan kode ini.

Tingkat kepercayaan:
- DOCX: `tingkat_kepercayaan = "tinggi"` — margin & font dibaca langsung
  dari properti dokumen (section.*_margin, run.font.*), bukan estimasi.
- PDF: `tingkat_kepercayaan = "rendah"` — PDF tidak menyimpan properti
  margin section. Margin PDF di sini adalah ESTIMASI dari bounding box
  konten teks per halaman (jarak terdekat teks ke tepi kertas), BUKAN
  nilai presisi dari pengaturan dokumen aslinya.


In [ ]:
PathLike = Union[str, Path]

# ---------------------------------------------------------------------------
# Util konversi & deteksi penomoran bab (independen dari Bagian A/B)
# ---------------------------------------------------------------------------


def _pt_ke_cm(pt: float) -> float:
    return pt / 72.0 * 2.54


def _length_ke_cm(nilai: Any) -> Optional[float]:
    """Mengonversi objek `Length` python-docx (EMU) ke cm. None-safe."""
    if nilai is None:
        return None
    return round(nilai.cm, 3)


_POLA_BAB = re.compile(r"^\s*bab\s+([ivxlcdm]+|\d+)\b", re.IGNORECASE)
_POLA_ROMAWI_VALID = re.compile(
    r"^M{0,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$", re.IGNORECASE
)


def _klasifikasi_gaya_nomor(token: str) -> str:
    """Mengklasifikasi token nomor bab ("I", "1", dst.) sebagai romawi/angka."""
    if token.isdigit():
        return "angka"
    if token and _POLA_ROMAWI_VALID.match(token):
        return "romawi"
    return "tidak_dikenali"


# ---------------------------------------------------------------------------
# Bagian DOCX
# ---------------------------------------------------------------------------


def _resolusi_font_run(run: Any, gaya_paragraf: Any) -> Dict[str, Any]:
    """Meresolusi nama & ukuran font EFEKTIF satu run docx.

    python-docx hanya mengembalikan nilai eksplisit yang di-override di
    level run (`run.font.name`/`run.font.size`); kalau None, nilainya
    diwariskan dari gaya paragraf, dan kalau gaya itu sendiri tidak
    menimpanya, diwariskan lagi dari `gaya.base_style`, dst. Fungsi ini
    menelusuri rantai itu SECARA TERPISAH untuk nama dan ukuran (karena
    keduanya bisa diresolusi dari level berbeda), dan mencatat level mana
    yang akhirnya menentukan nilai (`sumber_nama_font`/`sumber_ukuran_font`)
    supaya transparan untuk audit (mis. kasus style "Title" 22pt di atas).

    Return: {"nama_font", "sumber_nama_font", "ukuran_font_pt", "sumber_ukuran_font"}
    """
    nama_font = run.font.name
    sumber_nama = "run" if nama_font is not None else None
    ukuran_font = run.font.size.pt if run.font.size is not None else None
    sumber_ukuran = "run" if ukuran_font is not None else None

    gaya = gaya_paragraf
    tingkat = 0
    while (nama_font is None or ukuran_font is None) and gaya is not None:
        label = f"gaya:{gaya.name}" if tingkat == 0 else f"gaya_dasar:{gaya.name}"
        if nama_font is None and gaya.font.name is not None:
            nama_font = gaya.font.name
            sumber_nama = label
        if ukuran_font is None and gaya.font.size is not None:
            ukuran_font = gaya.font.size.pt
            sumber_ukuran = label
        gaya = gaya.base_style
        tingkat += 1

    return {
        "nama_font": nama_font,
        "sumber_nama_font": sumber_nama or "tidak_diketahui",
        "ukuran_font_pt": ukuran_font,
        "sumber_ukuran_font": sumber_ukuran or "tidak_diketahui",
    }


def ekstrak_metadata_docx(path: PathLike) -> Dict[str, Any]:
    """Mengekstrak metadata tata letak dari file .docx (jalur JM).

    Input:
        path: path ke file .docx.

    Output (dict):
        {
            "path_asal": str,
            "format_asli": "docx",
            "tingkat_kepercayaan": "tinggi",
            "section": [
                {
                    "indeks_section": int,
                    "margin_atas_cm": float | None, "margin_bawah_cm": float | None,
                    "margin_kiri_cm": float | None, "margin_kanan_cm": float | None,
                    "lebar_halaman_cm": float | None, "tinggi_halaman_cm": float | None,
                    "orientasi": "portrait" | "landscape" | None,
                },
                ...
            ],
            "font_per_paragraf": [   # HANYA paragraf yang punya run berteks (bukan kosong)
                {
                    "indeks_paragraf": int,       # sama dengan indexing `document.paragraphs` di Bagian A
                    "indeks_section": int | None,
                    "gaya": str | None,
                    "cuplikan_teks": str,          # 60 char pertama, MENTAH (belum dinormalisasi A)
                    "run": [
                        {
                            "indeks_run": int,
                            "nama_font": str | None, "sumber_nama_font": str,
                            "ukuran_font_pt": float | None, "sumber_ukuran_font": str,
                            "tebal": bool | None,   # nilai run.font.bold MENTAH (TIDAK diresolusi lewat rantai gaya)
                            "miring": bool | None,  # idem run.font.italic
                        },
                        ...
                    ],
                },
                ...
            ],
            "penomoran_bab_terdeteksi": [
                {"indeks_paragraf": int, "teks_tertangkap": str, "gaya_penomoran": "angka"|"romawi"|"tidak_dikenali"},
                ...
            ],
            "peringatan": [str, ...],
        }

    Catatan: `tebal`/`miring` sengaja TIDAK diresolusi lewat rantai gaya
    seperti nama/ukuran font (python-docx tidak expose ini semudah
    font.name/size), jadi bisa None meski secara visual tampak
    tebal/miring karena diwariskan dari gaya. Kalau butuh nilai efektif,
    itu perlu diresolusi terpisah oleh pemanggil lewat `gaya`/style chain.

    Keterbatasan lain: `_resolusi_font_run` hanya menelusuri rantai
    run -> style paragraf -> base_style. Kalau font TIDAK di-override di
    level manapun dalam rantai itu (diwariskan dari `docDefaults` pada
    styles.xml, mis. gaya "Normal" yang benar-benar polos), hasilnya
    `nama_font`/`ukuran_font_pt` = None dengan sumber "tidak_diketahui" —
    ini BUKAN berarti dokumen tidak punya font efektif (Word tetap
    merender sesuatu, mis. default Word Calibri 11), hanya berarti
    python-docx tidak expose docDefaults semudah style.font. Kalau kasus
    ini sering muncul di data riil, perlu penanganan tambahan (parsing
    manual `styles.xml` -> `w:docDefaults`) di luar cakupan modul ini.

    Raises:
        EkstraksiError: jika file tidak bisa dibuka.
    """
    try:
        from docx import Document
    except ImportError as exc:  # pragma: no cover
        raise EkstraksiError(
            "Paket 'python-docx' belum terpasang. Jalankan: pip install python-docx"
        ) from exc

    path = Path(path)
    peringatan: List[str] = []

    try:
        dokumen = Document(str(path))
    except Exception as exc:
        raise EkstraksiError(
            f"Gagal membuka file DOCX '{path.name}' untuk ekstraksi metadata JM: {exc}"
        ) from exc

    section_info: List[Dict[str, Any]] = []
    for i, section in enumerate(dokumen.sections):
        try:
            orientasi = section.orientation.name.lower()
        except Exception:
            orientasi = None
        section_info.append(
            {
                "indeks_section": i,
                "margin_atas_cm": _length_ke_cm(section.top_margin),
                "margin_bawah_cm": _length_ke_cm(section.bottom_margin),
                "margin_kiri_cm": _length_ke_cm(section.left_margin),
                "margin_kanan_cm": _length_ke_cm(section.right_margin),
                "lebar_halaman_cm": _length_ke_cm(section.page_width),
                "tinggi_halaman_cm": _length_ke_cm(section.page_height),
                "orientasi": orientasi,
            }
        )

    indeks_section_per_paragraf = _petakan_indeks_section_docx(dokumen)

    font_per_paragraf: List[Dict[str, Any]] = []
    penomoran_bab: List[Dict[str, Any]] = []

    for i, paragraf in enumerate(dokumen.paragraphs):
        teks_baris = paragraf.text.strip()

        cocok_bab = _POLA_BAB.match(teks_baris)
        if cocok_bab:
            penomoran_bab.append(
                {
                    "indeks_paragraf": i,
                    "teks_tertangkap": teks_baris,
                    "gaya_penomoran": _klasifikasi_gaya_nomor(cocok_bab.group(1)),
                }
            )

        run_info: List[Dict[str, Any]] = []
        for j, run in enumerate(paragraf.runs):
            if not run.text.strip():
                continue
            resolusi = _resolusi_font_run(run, paragraf.style)
            run_info.append(
                {
                    "indeks_run": j,
                    "tebal": run.font.bold,
                    "miring": run.font.italic,
                    **resolusi,
                }
            )

        if not run_info:
            continue

        gaya_nama = None
        try:
            if paragraf.style is not None:
                gaya_nama = paragraf.style.name
        except Exception:
            gaya_nama = None

        font_per_paragraf.append(
            {
                "indeks_paragraf": i,
                "indeks_section": (
                    indeks_section_per_paragraf[i] if i < len(indeks_section_per_paragraf) else None
                ),
                "gaya": gaya_nama,
                "cuplikan_teks": teks_baris[:60],
                "run": run_info,
            }
        )

    if len(section_info) > 1:
        peringatan.append(
            f"Dokumen memiliki {len(section_info)} section dengan kemungkinan margin "
            "berbeda-beda antar section (lihat temuan template resmi); JANGAN asumsikan "
            "satu margin berlaku untuk seluruh dokumen. Gunakan field 'indeks_section' "
            "pada tiap entri untuk mencocokkan margin yang relevan."
        )

    return {
        "path_asal": str(path),
        "format_asli": "docx",
        "tingkat_kepercayaan": "tinggi",
        "section": section_info,
        "font_per_paragraf": font_per_paragraf,
        "penomoran_bab_terdeteksi": penomoran_bab,
        "peringatan": peringatan,
    }


# ---------------------------------------------------------------------------
# Bagian PDF
# ---------------------------------------------------------------------------


def _ambil_baris_dan_span_halaman(halaman: Any) -> List[Dict[str, Any]]:
    """Mengambil baris teks PDF beserta span font-nya LANGSUNG dari file asli.

    Sengaja terpisah dari `_ambil_baris_halaman` di Bagian A (yang hanya
    butuh teks per baris untuk pembersihan header/footer) karena di sini
    granularitas SPAN (bukan baris) yang dibutuhkan — satu baris bisa
    berisi beberapa span dengan font berbeda.

    Return: list of {"teks": str, "span": [{"nama_font", "ukuran_font_pt",
        "tebal", "miring", "x0", "y0", "x1", "y1"}, ...]}
    """
    data = halaman.get_text("dict")
    hasil: List[Dict[str, Any]] = []
    for blok in data.get("blocks", []):
        for baris in blok.get("lines", []):
            span_list: List[Dict[str, Any]] = []
            potongan_teks: List[str] = []
            for span in baris.get("spans", []):
                teks_span = span.get("text", "")
                if not teks_span.strip():
                    continue
                potongan_teks.append(teks_span)
                flags = span.get("flags", 0)
                nama_font = span.get("font")
                x0, y0, x1, y1 = span.get("bbox", (0.0, 0.0, 0.0, 0.0))
                span_list.append(
                    {
                        "nama_font": nama_font,
                        "ukuran_font_pt": round(float(span.get("size", 0.0)), 2),
                        # flags bit 4 (0x10) = bold, bit 1 (0x2) = italic (dok. PyMuPDF);
                        # ditambah cek nama font sbg jaring pengaman krn flags kadang tidak akurat
                        # tergantung aplikasi pembuat PDF.
                        "tebal": bool(flags & 0x10) or "bold" in (nama_font or "").lower(),
                        "miring": bool(flags & 0x2)
                        or any(k in (nama_font or "").lower() for k in ("italic", "oblique")),
                        "x0": x0,
                        "y0": y0,
                        "x1": x1,
                        "y1": y1,
                    }
                )
            if span_list:
                hasil.append({"teks": "".join(potongan_teks).strip(), "span": span_list})
    return hasil


def _perkirakan_margin_halaman(
    span_list: List[Dict[str, Any]], lebar_pt: float, tinggi_pt: float
) -> Dict[str, Optional[float]]:
    """Mengestimasi margin dari bounding box konten teks per halaman.

    Ini BUKAN margin presisi (PDF tidak menyimpan properti margin section
    seperti .docx) — hanya jarak terdekat teks yang terdeteksi ke tepi
    kertas. Bisa meleset kalau halaman punya sedikit konten (margin
    ter-estimasi lebih besar dari aslinya) atau ada elemen non-teks
    (gambar/tabel garis) yang sebenarnya lebih dekat ke tepi.
    """
    if not span_list:
        return {
            "margin_atas_cm": None,
            "margin_bawah_cm": None,
            "margin_kiri_cm": None,
            "margin_kanan_cm": None,
        }
    margin_kiri_pt = min(s["x0"] for s in span_list)
    margin_kanan_pt = lebar_pt - max(s["x1"] for s in span_list)
    margin_atas_pt = min(s["y0"] for s in span_list)
    margin_bawah_pt = tinggi_pt - max(s["y1"] for s in span_list)
    return {
        "margin_atas_cm": round(_pt_ke_cm(margin_atas_pt), 3),
        "margin_bawah_cm": round(_pt_ke_cm(margin_bawah_pt), 3),
        "margin_kiri_cm": round(_pt_ke_cm(margin_kiri_pt), 3),
        "margin_kanan_cm": round(_pt_ke_cm(margin_kanan_pt), 3),
    }


def ekstrak_metadata_pdf(path: PathLike) -> Dict[str, Any]:
    """Mengekstrak metadata tata letak dari file .pdf (jalur JM), berupa ESTIMASI.

    Input:
        path: path ke file .pdf.

    Output (dict):
        {
            "path_asal": str,
            "format_asli": "pdf",
            "tingkat_kepercayaan": "rendah",
            "halaman": [
                {
                    "indeks_halaman": int,
                    "lebar_halaman_cm": float, "tinggi_halaman_cm": float,
                    "margin_atas_cm": float | None, "margin_bawah_cm": float | None,
                    "margin_kiri_cm": float | None, "margin_kanan_cm": float | None,
                },
                ...
            ],
            "font_per_halaman": [
                {"indeks_halaman": int, "span": [{"nama_font", "ukuran_font_pt", "tebal", "miring", "x0","y0","x1","y1"}, ...]},
                ...
            ],
            "penomoran_bab_terdeteksi": [
                {"indeks_halaman": int, "teks_tertangkap": str, "gaya_penomoran": str},
                ...
            ],
            "peringatan": [str, ...],   # SELALU berisi disclaimer estimasi margin
        }

    Raises:
        EkstraksiError: jika file tidak bisa dibuka atau tidak punya halaman.
    """
    try:
        import pymupdf
    except ImportError as exc:  # pragma: no cover
        raise EkstraksiError(
            "Paket 'PyMuPDF' belum terpasang. Jalankan: pip install PyMuPDF"
        ) from exc

    path = Path(path)
    peringatan: List[str] = [
        "Estimasi margin PDF dihitung dari bounding box konten teks per halaman, "
        "BUKAN nilai margin presisi dari pengaturan dokumen aslinya (PDF tidak "
        "menyimpan properti margin section seperti .docx)."
    ]

    try:
        dokumen = pymupdf.open(str(path))
    except Exception as exc:
        raise EkstraksiError(
            f"Gagal membuka file PDF '{path.name}' untuk ekstraksi metadata JM: {exc}"
        ) from exc

    try:
        jumlah_halaman = dokumen.page_count
        if jumlah_halaman == 0:
            raise EkstraksiError(f"File PDF '{path.name}' tidak memiliki halaman.")

        halaman_info: List[Dict[str, Any]] = []
        font_per_halaman: List[Dict[str, Any]] = []
        penomoran_bab: List[Dict[str, Any]] = []
        ada_span_sama_sekali = False

        for idx_halaman, halaman in enumerate(dokumen):
            lebar = float(halaman.rect.width)
            tinggi = float(halaman.rect.height)
            baris_span = _ambil_baris_dan_span_halaman(halaman)
            semua_span = [s for baris in baris_span for s in baris["span"]]
            if semua_span:
                ada_span_sama_sekali = True

            margin_estimasi = _perkirakan_margin_halaman(semua_span, lebar, tinggi)
            halaman_info.append(
                {
                    "indeks_halaman": idx_halaman,
                    "lebar_halaman_cm": round(_pt_ke_cm(lebar), 3),
                    "tinggi_halaman_cm": round(_pt_ke_cm(tinggi), 3),
                    **margin_estimasi,
                }
            )
            font_per_halaman.append({"indeks_halaman": idx_halaman, "span": semua_span})

            for baris in baris_span:
                cocok = _POLA_BAB.match(baris["teks"])
                if cocok:
                    penomoran_bab.append(
                        {
                            "indeks_halaman": idx_halaman,
                            "teks_tertangkap": baris["teks"],
                            "gaya_penomoran": _klasifikasi_gaya_nomor(cocok.group(1)),
                        }
                    )

        if not ada_span_sama_sekali:
            peringatan.append(
                "Tidak ada span teks yang berhasil diambil dari file ini (kemungkinan "
                "hasil pindai/scan); metadata font & estimasi margin tidak tersedia "
                "untuk halaman manapun."
            )

        return {
            "path_asal": str(path),
            "format_asli": "pdf",
            "tingkat_kepercayaan": "rendah",
            "halaman": halaman_info,
            "font_per_halaman": font_per_halaman,
            "penomoran_bab_terdeteksi": penomoran_bab,
            "peringatan": peringatan,
        }
    finally:
        dokumen.close()


# ---------------------------------------------------------------------------
# Fungsi pemilih format
# ---------------------------------------------------------------------------


def ekstrak_metadata_dokumen(path: PathLike) -> Dict[str, Any]:
    """Mendeteksi format dari ekstensi file lalu memanggil ekstraktor metadata JM yang sesuai."""
    path = Path(path)
    ekstensi = path.suffix.lower()
    if ekstensi == ".docx":
        return ekstrak_metadata_docx(path)
    if ekstensi == ".pdf":
        return ekstrak_metadata_pdf(path)
    raise EkstraksiError(
        f"Format file tidak didukung untuk '{path.name}' (ekstensi '{ekstensi}'). "
        "Hanya .docx dan .pdf yang didukung."
    )


## Bagian D — Pemetaan Ground Truth

Dua tanggung jawab:
1. `muat_ground_truth`: membaca & memvalidasi sheet "Anotasi" dari workbook
   ground truth (proposal_id, unit_id, kode_pelanggaran, dst.).
2. `gabungkan_teks_dan_ground_truth`: menggabungkan hasil segmentasi teks
   Bagian B (+ metadata JM Bagian C bila relevan) dengan ground truth,
   menghasilkan satu record per (proposal_id, unit_id).

PEMBARUAN (2026-09-18) — sumber kebenaran terkonfirmasi:
Setelah memeriksa langsung workbook ground truth asli peneliti (sheet
"Referensi" dan "Anotasi_Gabungan"), dua hal di bawah ini SUDAH TIDAK
lagi tebakan, melainkan dikonfirmasi dari data asli:
- `KODE_TAKSONOMI_AKTIF_DEFAULT`: 21 kode taksonomi yang benar-benar aktif
  (dikutip dari sheet "Referensi", kolom "DAFTAR KODE PELANGGARAN", status
  "AKTIF" — mengecualikan FMT-03 & BHS-02 yang eksplisit ditandai
  dinonaktifkan/"gunakan NA-03"). Ini sekarang jadi DEFAULT `muat_ground_truth`,
  bukan lagi opsional.
- `PETA_KOLOM_ANOTASI_DEFAULT` & `nama_sheet="Anotasi_Gabungan"`: nama
  kolom dan nama sheet di bawah sekarang mencerminkan struktur ASLI
  workbook peneliti (proposal_id, unit_id, kode_pelanggaran, bukti,
  dasar_pedoman, keyakinan, anotator, catatan, status_adjudikasi).

Catatan: ini dikonfirmasi untuk WORKBOOK SPESIFIK peneliti ini (skripsi
ini), bukan klaim skema 11-kolom resmi Panduan Anotasi v0.2 berlaku
universal. Kalau workbook lain/versi lebih baru punya struktur berbeda,
tetap gunakan parameter `peta_kolom`/`nama_sheet`/`kode_taksonomi_valid`
untuk menyesuaikan — jangan asumsikan default ini akan selalu cocok.

Asumsi lain:
- Satu baris di sheet "Anotasi" = satu instans anotasi untuk satu
  (proposal_id, unit_id); kolom `kode_pelanggaran` boleh berisi lebih dari
  satu kode dalam satu sel (dipisah koma/titik-koma).
- Kalau tidak ada baris ground truth sama sekali untuk suatu
  (proposal_id, unit_id), itu DICATAT sebagai status "TIDAK_ADA_ANOTASI"
  (bukan diam-diam dilewati), karena bisa berarti unit itu belum
  dianotasi ATAU memang tidak relevan — perlu ditinjau manusia.


In [ ]:
PathLike = Union[str, Path]


class GroundTruthError(EkstraksiError):
    """Kesalahan saat memuat/memvalidasi berkas ground truth.

    Subclass dari `EkstraksiError` (Bagian A) supaya pemanggil batch
    (Bagian G) bisa menangani kegagalan di modul A-D lewat satu jenis
    exception yang sama tanpa menghentikan seluruh proses.
    """


# ---------------------------------------------------------------------------
# Skema kolom & kode (lihat catatan kalibrasi di docstring modul)
# ---------------------------------------------------------------------------

PETA_KOLOM_ANOTASI_DEFAULT: Dict[str, str] = {
    "proposal_id": "proposal_id",
    "unit_id": "unit_id",
    "kode_pelanggaran": "kode_pelanggaran",
    "jalur_deteksi": "jalur_deteksi",
    "dasar_pedoman": "dasar_pedoman",
    "tanggal_anotasi": "tanggal_anotasi",
    "catatan": "catatan",
    # Ditambahkan setelah memeriksa workbook asli (lihat "PEMBARUAN" di
    # docstring modul) — semua opsional, dilewati kalau kolomnya tidak ada.
    "bukti": "bukti",
    "keyakinan": "keyakinan",
    "anotator": "anotator",
    "status_adjudikasi": "status_adjudikasi",
}

_KOLOM_WAJIB = ("proposal_id", "unit_id", "kode_pelanggaran")

# Dikutip langsung dari sheet "Referensi" workbook ground truth peneliti
# (kolom "DAFTAR KODE PELANGGARAN", status "AKTIF"). FMT-03 dan BHS-02
# SENGAJA tidak disertakan -- keduanya ditandai eksplisit "dinonaktifkan,
# gunakan NA-03" di sheet yang sama. frozenset supaya aman dipakai sbg
# nilai default parameter (immutable, tidak kena masalah mutable default).
KODE_TAKSONOMI_AKTIF_DEFAULT: FrozenSet[str] = frozenset(
    {
        "KEL-01", "KEL-02", "KEL-03", "KEL-04", "KEL-05", "KEL-06",
        "ISI-01", "ISI-02", "ISI-03",
        "ANG-01", "ANG-02",
        "KON-01", "KON-02", "KON-03", "KON-04",
        "ADM-01", "ADM-02", "ADM-04",
        "FMT-01", "FMT-02", "FMT-04",
        "BHS-01",
    }
)

_STATUS_KHUSUS: Set[str] = {"SESUAI", "NA-01", "NA-02", "NA-03"}
_POLA_KODE_TAKSONOMI = re.compile(r"^(KEL|ISI|ANG|KON|ADM|FMT|BHS)-\d{2}$")
_POLA_JALUR_DETEKSI = re.compile(r"^(JP|JC|JD|JM)$", re.IGNORECASE)
_POLA_DELIMITER_MULTI_NILAI = re.compile(r"[,;]\s*")


def _adalah_kode_valid(kode: str, kode_taksonomi_valid: Optional[AbstractSet[str]]) -> bool:
    """True jika `kode` adalah status khusus, ATAU kode taksonomi yang dikenali.

    Kalau `kode_taksonomi_valid` diberikan, dipakai sebagai keanggotaan set
    yang ketat (sumber kebenaran dari peneliti). Kalau tidak, jatuh ke
    validasi pola generik (lihat catatan kalibrasi di docstring modul).
    """
    if kode in _STATUS_KHUSUS:
        return True
    if kode_taksonomi_valid is not None:
        return kode in kode_taksonomi_valid
    return bool(_POLA_KODE_TAKSONOMI.match(kode))


def _pecah_nilai_multi(nilai: Any) -> List[str]:
    """Memecah satu sel yang mungkin berisi beberapa nilai dipisah koma/titik-koma."""
    if nilai is None or (isinstance(nilai, float) and pd.isna(nilai)) or pd.isna(nilai):
        return []
    teks = str(nilai).strip()
    if not teks:
        return []
    return [bagian.strip() for bagian in _POLA_DELIMITER_MULTI_NILAI.split(teks) if bagian.strip()]


def _ambil_nilai_tunggal(nilai: Any) -> Optional[str]:
    """Mengambil satu sel sebagai string tunggal APA ADANYA (TIDAK dipecah
    koma/titik-koma seperti `_pecah_nilai_multi`) -- dipakai untuk kolom
    teks bebas/kategorikal (mis. `bukti`, `keyakinan`) yang bisa secara sah
    mengandung koma sbg tanda baca biasa, bukan pemisah multi-nilai.
    """
    if nilai is None or pd.isna(nilai):
        return None
    teks = str(nilai).strip()
    return teks or None


# ---------------------------------------------------------------------------
# D1: muat_ground_truth
# ---------------------------------------------------------------------------


def muat_ground_truth(
    path_excel: PathLike,
    peta_kolom: Optional[Dict[str, str]] = None,
    kode_taksonomi_valid: Optional[AbstractSet[str]] = KODE_TAKSONOMI_AKTIF_DEFAULT,
    nama_sheet: str = "Anotasi_Gabungan",
) -> pd.DataFrame:
    """Memuat & memvalidasi sheet ground truth dari workbook Excel.

    Input:
        path_excel: path ke workbook ground truth (.xlsx).
        peta_kolom: pemetaan nama kolom logis -> nama kolom aktual di file
            Anda (mengganti sebagian/seluruh `PETA_KOLOM_ANOTASI_DEFAULT`).
            Default-nya sudah cocok utk workbook peneliti (lihat "PEMBARUAN"
            di docstring modul); sesuaikan kalau workbook Anda berbeda.
        kode_taksonomi_valid: set kode taksonomi aktif yang dipakai untuk
            validasi keanggotaan ketat. Default `KODE_TAKSONOMI_AKTIF_DEFAULT`
            (21 kode aktif asli, lihat docstring modul). Pass `None` secara
            eksplisit untuk kembali ke validasi pola generik yang lebih
            longgar (kalau memang perlu memproses workbook dgn daftar kode
            berbeda yang belum diketahui).
        nama_sheet: nama sheet ground truth di workbook (default
            "Anotasi_Gabungan", sesuai workbook peneliti).

    Output:
        pandas.DataFrame — semua baris asli TETAP disertakan apa adanya
        (baris bermasalah TIDAK dibuang diam-diam), ditambah 2 kolom:
        - "_baris_excel": nomor baris asli di file Excel (1-based, sudah
          memperhitungkan baris header), untuk audit manual.
        - "_masalah_validasi": list[str] berisi pesan masalah validasi
          baris tsb (list kosong kalau baris valid).
        Ringkasan agregat validasi tersimpan di `DataFrame.attrs["ringkasan_validasi"]`
        (dict: sheet, jumlah_baris, jumlah_baris_bermasalah, kolom_dipakai).

    Validasi per baris:
        - unit_id harus salah satu dari 16 unit (U00-U15), tidak
          peka-huruf-besar/kecil untuk pengecekan (nilai asli tidak diubah).
        - kode_pelanggaran (boleh multi-nilai dipisah koma/titik-koma) harus
          SESUAI/NA-01/NA-02/NA-03 atau lolos `_adalah_kode_valid`.
        - jalur_deteksi (kalau kolomnya ada) harus salah satu JP/JC/JD/JM.
        - tanggal_anotasi (kalau kolomnya ada) harus bisa diparse sebagai
          tanggal (dayfirst=True, sesuai konvensi Indonesia).

    Raises:
        GroundTruthError: file/sheet tidak bisa dibaca, atau kolom WAJIB
            (proposal_id, unit_id, kode_pelanggaran — setelah `peta_kolom`
            diterapkan) tidak ditemukan di sheet.
    """
    peta = {**PETA_KOLOM_ANOTASI_DEFAULT, **(peta_kolom or {})}
    path_excel = Path(path_excel)

    try:
        # dtype=str: semua kolom dibaca sbg teks apa adanya supaya regex
        # validasi & pemecahan multi-nilai konsisten (tanggal tetap bisa
        # diparse ulang lewat pd.to_datetime di bawah).
        df = pd.read_excel(path_excel, sheet_name=nama_sheet, engine="openpyxl", dtype=str)
    except Exception as exc:
        raise GroundTruthError(
            f"Gagal membaca sheet '{nama_sheet}' dari '{path_excel.name}': {exc}"
        ) from exc

    kolom_wajib_hilang = [peta[k] for k in _KOLOM_WAJIB if peta[k] not in df.columns]
    if kolom_wajib_hilang:
        raise GroundTruthError(
            f"Kolom wajib tidak ditemukan di sheet '{nama_sheet}': {kolom_wajib_hilang}. "
            "Kalau nama kolom di file Excel Anda berbeda, gunakan parameter `peta_kolom` "
            "untuk memetakan nama kolom aktual (lihat PETA_KOLOM_ANOTASI_DEFAULT)."
        )

    df = df.reset_index(drop=True)
    df["_baris_excel"] = df.index + 2  # +2: 1-based, baris 1 = header

    masalah_per_baris: List[List[str]] = []
    for _, baris in df.iterrows():
        masalah: List[str] = []

        unit_id_mentah = baris.get(peta["unit_id"])
        unit_id_norm = str(unit_id_mentah).strip().upper() if pd.notna(unit_id_mentah) else ""
        if unit_id_norm not in ID_UNIT_VALID:
            masalah.append(f"unit_id tidak valid: '{unit_id_mentah}'")

        daftar_kode = _pecah_nilai_multi(baris.get(peta["kode_pelanggaran"]))
        if not daftar_kode:
            masalah.append("kode_pelanggaran kosong")
        else:
            for kode in daftar_kode:
                if not _adalah_kode_valid(kode.upper(), kode_taksonomi_valid):
                    masalah.append(f"kode_pelanggaran tidak dikenali: '{kode}'")

        kolom_jalur = peta.get("jalur_deteksi")
        if kolom_jalur and kolom_jalur in df.columns:
            nilai_jalur = baris.get(kolom_jalur)
            if pd.notna(nilai_jalur) and str(nilai_jalur).strip():
                if not _POLA_JALUR_DETEKSI.match(str(nilai_jalur).strip()):
                    masalah.append(f"jalur_deteksi tidak dikenali: '{nilai_jalur}' (harus JP/JC/JD/JM)")

        kolom_tanggal = peta.get("tanggal_anotasi")
        if kolom_tanggal and kolom_tanggal in df.columns:
            nilai_tanggal = baris.get(kolom_tanggal)
            if pd.notna(nilai_tanggal) and str(nilai_tanggal).strip():
                tanggal_terparse = pd.to_datetime(nilai_tanggal, errors="coerce", dayfirst=True)
                if pd.isna(tanggal_terparse):
                    masalah.append(f"tanggal_anotasi tidak valid: '{nilai_tanggal}'")

        masalah_per_baris.append(masalah)

    df["_masalah_validasi"] = masalah_per_baris

    jumlah_bermasalah = sum(1 for m in masalah_per_baris if m)
    df.attrs["ringkasan_validasi"] = {
        "sheet": nama_sheet,
        "jumlah_baris": len(df),
        "jumlah_baris_bermasalah": jumlah_bermasalah,
        "kolom_dipakai": peta,
    }

    return df


# ---------------------------------------------------------------------------
# D2: gabungkan_teks_dan_ground_truth
# ---------------------------------------------------------------------------


def _ringkas_status_dan_kode(semua_nilai: List[str]) -> Tuple[str, List[str]]:
    """Meringkas semua nilai kode_pelanggaran (gabungan seluruh baris ground
    truth) untuk satu (proposal_id, unit_id) menjadi satu status ringkas +
    daftar kode pelanggaran nyata (SESUAI/NA-0X TIDAK termasuk di daftar
    ini karena bukan kode pelanggaran, hanya status).

    Prioritas: ada kode pelanggaran nyata (pola taksonomi) > NA-01/02/03 >
    SESUAI > tidak ada anotasi sama sekali.
    """
    semua_nilai_upper = [v.upper() for v in semua_nilai]

    kode_nyata: List[str] = []
    terlihat: Set[str] = set()
    for v in semua_nilai_upper:
        if _POLA_KODE_TAKSONOMI.match(v) and v not in terlihat:
            kode_nyata.append(v)
            terlihat.add(v)
    if kode_nyata:
        return "PELANGGARAN", kode_nyata

    for status_na in ("NA-01", "NA-02", "NA-03"):
        if status_na in semua_nilai_upper:
            return status_na, []

    if "SESUAI" in semua_nilai_upper:
        return "SESUAI", []

    return "TIDAK_ADA_ANOTASI", []


def gabungkan_teks_dan_ground_truth(
    hasil_segmentasi: Dict[str, Any],
    gt_dataframe: pd.DataFrame,
    proposal_id: str,
    metadata_jm: Optional[Dict[str, Any]] = None,
    bahasa_asli: Optional[str] = None,
    format_asli: Optional[str] = None,
    peta_kolom: Optional[Dict[str, str]] = None,
) -> List[Dict[str, Any]]:
    """Menggabungkan teks unit (Bagian B) + ground truth (D1) jadi 1 record/unit.

    Input:
        hasil_segmentasi: keluaran `segmentasi_unit()` (Bagian B) — dict
            `{"unit": {unit_id: {...}}, "peringatan": [...]}`. Dict polos
            `{unit_id: {"teks": ...}}` juga diterima untuk fleksibilitas.
        gt_dataframe: keluaran `muat_ground_truth()` (D1).
        proposal_id: ID proposal yang sedang diproses (dicocokkan ke kolom
            proposal_id di `gt_dataframe`, dibandingkan sbg string setelah di-strip).
        metadata_jm: dict keluaran `ekstrak_metadata_docx`/`ekstrak_metadata_pdf`
            (Bagian C). Hanya dilekatkan ke record U00 (unit lain -> None),
            karena hanya U00 yang merepresentasikan properti tata letak
            keseluruhan dokumen.
        bahasa_asli: "ID"/"EN" dari metadata proposal (sheet Daftar_Proposal).
        format_asli: "docx"/"pdf" (biasanya `hasil_ekstraksi["format_asli"]` dari Bagian A).
        peta_kolom: sama seperti di `muat_ground_truth` — HARUS konsisten
            dengan peta_kolom yang dipakai saat memanggil `muat_ground_truth`,
            supaya nama kolom yang dicocokkan sama.

    Output:
        list[dict], SELALU 16 record (satu per unit U00-U15, urutan
        `DAFTAR_UNIT`), masing-masing:
        {
            "proposal_id": str,
            "unit_id": str,
            "bahasa_asli": str | None,
            "format_asli": str | None,
            "teks_unit": str,
            "status_ground_truth": "PELANGGARAN"|"SESUAI"|"NA-01"|"NA-02"|"NA-03"|"TIDAK_ADA_ANOTASI",
            "kode_pelanggaran": list[str],      # kosong kalau bukan PELANGGARAN
            "dasar_pedoman": list[str],         # rujukan Bab/Pasal/ayat, gabungan semua baris terkait
            "bukti": list[str],                  # kutipan/evidence per baris anotasi terkait unit ini
            "keyakinan": list[str],              # mis. "Tinggi"/"Sedang"/"Rendah", satu per baris
            "anotator": list[str],               # mis. "A1"/"A2"/"A1 + A2", satu per baris
            "status_adjudikasi": list[str],      # mis. "Disepakati"/"Diubah", satu per baris (kalau kolomnya ada & terisi)
                                                  # ^ keempat list di atas (dasar_pedoman s.d. status_adjudikasi)
                                                  # dikumpulkan APA ADANYA per baris ground truth yang cocok,
                                                  # TIDAK dijamin berpasangan 1:1 dgn tiap kode di 'kode_pelanggaran'
                                                  # kalau satu baris berisi banyak kode sekaligus dalam satu sel.
            "metadata_jm": dict | None,          # hanya diisi utk unit_id == "U00"
            "catatan_ekstraksi": str | None,     # gabungan warning Bagian B + masalah validasi GT
        }

    Raises:
        GroundTruthError: kalau `gt_dataframe` tidak punya kolom proposal_id/unit_id
            yang dipetakan (mis. dipanggil dgn DataFrame yang bukan dari `muat_ground_truth`).
    """
    peta = {**PETA_KOLOM_ANOTASI_DEFAULT, **(peta_kolom or {})}
    kolom_proposal_id = peta["proposal_id"]
    kolom_unit_id = peta["unit_id"]
    kolom_kode = peta["kode_pelanggaran"]
    kolom_dasar = peta.get("dasar_pedoman")
    kolom_bukti = peta.get("bukti")
    kolom_keyakinan = peta.get("keyakinan")
    kolom_anotator = peta.get("anotator")
    kolom_status_adjudikasi = peta.get("status_adjudikasi")

    for kolom in (kolom_proposal_id, kolom_unit_id, kolom_kode):
        if kolom not in gt_dataframe.columns:
            raise GroundTruthError(
                f"Kolom '{kolom}' tidak ditemukan di gt_dataframe. Pastikan `peta_kolom` "
                "yang dipakai di sini sama dengan yang dipakai saat `muat_ground_truth()`."
            )

    unit_map: Dict[str, Any] = hasil_segmentasi.get("unit", hasil_segmentasi)

    baris_proposal = gt_dataframe[
        gt_dataframe[kolom_proposal_id].astype(str).str.strip() == str(proposal_id).strip()
    ]

    hasil: List[Dict[str, Any]] = []
    for unit_id, _nama_unit in DAFTAR_UNIT:
        info_unit = unit_map.get(unit_id, {}) or {}
        teks_unit = info_unit.get("teks", "")
        catatan_segmentasi = info_unit.get("catatan")

        baris_unit = baris_proposal[
            baris_proposal[kolom_unit_id].astype(str).str.strip().str.upper() == unit_id
        ]

        semua_nilai_kode: List[str] = []
        daftar_dasar: List[str] = []
        daftar_bukti: List[str] = []
        daftar_keyakinan: List[str] = []
        daftar_anotator: List[str] = []
        daftar_status_adjudikasi: List[str] = []
        masalah_gt_unit: List[str] = []
        for _, baris in baris_unit.iterrows():
            semua_nilai_kode.extend(_pecah_nilai_multi(baris.get(kolom_kode)))
            if kolom_dasar and kolom_dasar in gt_dataframe.columns:
                daftar_dasar.extend(_pecah_nilai_multi(baris.get(kolom_dasar)))
            if kolom_bukti and kolom_bukti in gt_dataframe.columns:
                nilai = _ambil_nilai_tunggal(baris.get(kolom_bukti))
                if nilai:
                    daftar_bukti.append(nilai)
            if kolom_keyakinan and kolom_keyakinan in gt_dataframe.columns:
                nilai = _ambil_nilai_tunggal(baris.get(kolom_keyakinan))
                if nilai:
                    daftar_keyakinan.append(nilai)
            if kolom_anotator and kolom_anotator in gt_dataframe.columns:
                nilai = _ambil_nilai_tunggal(baris.get(kolom_anotator))
                if nilai:
                    daftar_anotator.append(nilai)
            if kolom_status_adjudikasi and kolom_status_adjudikasi in gt_dataframe.columns:
                nilai = _ambil_nilai_tunggal(baris.get(kolom_status_adjudikasi))
                if nilai:
                    daftar_status_adjudikasi.append(nilai)
            if "_masalah_validasi" in gt_dataframe.columns:
                masalah_gt_unit.extend(baris.get("_masalah_validasi") or [])

        status_gt, kode_pelanggaran = _ringkas_status_dan_kode(semua_nilai_kode)
        daftar_dasar_unik = list(dict.fromkeys(daftar_dasar))
        daftar_bukti_unik = list(dict.fromkeys(daftar_bukti))
        daftar_keyakinan_unik = list(dict.fromkeys(daftar_keyakinan))
        daftar_anotator_unik = list(dict.fromkeys(daftar_anotator))
        daftar_status_adjudikasi_unik = list(dict.fromkeys(daftar_status_adjudikasi))

        catatan_bagian: List[str] = []
        if catatan_segmentasi:
            catatan_bagian.append(catatan_segmentasi)
        if baris_unit.empty:
            catatan_bagian.append(
                f"Tidak ada baris anotasi ground truth untuk (proposal_id={proposal_id}, unit_id={unit_id})."
            )
        if masalah_gt_unit:
            catatan_bagian.append("Masalah validasi ground truth: " + "; ".join(masalah_gt_unit))

        hasil.append(
            {
                "proposal_id": proposal_id,
                "unit_id": unit_id,
                "bahasa_asli": bahasa_asli,
                "format_asli": format_asli,
                "teks_unit": teks_unit,
                "status_ground_truth": status_gt,
                "kode_pelanggaran": kode_pelanggaran,
                "dasar_pedoman": daftar_dasar_unik,
                "bukti": daftar_bukti_unik,
                "keyakinan": daftar_keyakinan_unik,
                "anotator": daftar_anotator_unik,
                "status_adjudikasi": daftar_status_adjudikasi_unik,
                "metadata_jm": metadata_jm if unit_id == "U00" else None,
                "catatan_ekstraksi": "; ".join(catatan_bagian) if catatan_bagian else None,
            }
        )

    return hasil


## Bagian E — Statistik Dataset

Modul ini menghitung ringkasan statistik dari kumpulan record hasil Bagian D
(`gabungkan_teks_dan_ground_truth`), DIGABUNG LINTAS SELURUH PROPOSAL dalam
dataset — dipakai untuk melaporkan profil dataset ke pembimbing: jumlah
proposal, distribusi pelanggaran per kode/kategori/unit, dan kode taksonomi
dengan instans terlalu sedikit untuk dianalisis terpisah (kandidat digabung
kategori "lain-lain").

Asumsi:
- `list_record` adalah gabungan list[dict] dari SEMUA proposal (hasil
  `gabungkan_teks_dan_ground_truth` dipanggil berulang per proposal lalu
  di-`extend()` jadi satu list besar), BUKAN hanya satu proposal.
- `kode_pelanggaran` pada tiap record HANYA berisi kode taksonomi asli
  (mis. "ISI-02"), bukan status seperti SESUAI/NA-0X — ini dijamin oleh
  `_ringkas_status_dan_kode` di Bagian D — sehingga kategori (KEL/ISI/ANG/
  KON/ADM/FMT/BHS) bisa diambil langsung dari prefiks sebelum "-".
- Statistik bahasa_asli/format_asli dihitung PER PROPOSAL (bukan per unit),
  supaya tidak "dobel-hitung" hingga 16x untuk proposal yang sama (tiap
  proposal punya 16 record, satu per unit, dengan bahasa_asli/format_asli
  yang identik di semua unit-nya).


In [ ]:
def _kategori_dari_kode(kode: str) -> str:
    """Mengambil kategori taksonomi (KEL/ISI/ANG/KON/ADM/FMT/BHS) dari kode, mis. "ISI-02" -> "ISI"."""
    return kode.split("-", 1)[0] if "-" in kode else "TIDAK_DIKETAHUI"


def hitung_statistik(
    list_record: List[Dict[str, Any]],
    ambang_instans_rendah: int = 8,
) -> Dict[str, Any]:
    """Menghitung ringkasan statistik dataset dari kumpulan record Bagian D.

    Input:
        list_record: gabungan record `gabungkan_teks_dan_ground_truth` lintas
            SEMUA proposal dalam dataset (bukan satu proposal saja).
        ambang_instans_rendah: batas jumlah instans (EKSKLUSIF) — kode
            dengan jumlah instans < ambang ini ditandai sebagai kandidat
            digabung ke kategori "lain-lain" (default 8, sesuai kebutuhan
            revisi C2).

    Output (dict):
        {
            "jumlah_proposal": int,                     # jumlah proposal_id unik
            "jumlah_unit_total": int,                   # = len(list_record)
            "jumlah_pelanggaran_per_kode": {kode: int},  # terurut menurun berdasar jumlah, lalu abjad
            "rata_rata_pelanggaran_per_proposal": float,
            "kode_instans_rendah": [kode, ...],          # jumlah < ambang_instans_rendah, terurut abjad

            # --- statistik tambahan, membantu profil dataset utk revisi C2 ---
            "jumlah_pelanggaran_per_kategori": {kategori: int},
            "jumlah_pelanggaran_per_unit": {unit_id: int},
            "distribusi_status_ground_truth": {status: int},   # dihitung per record (proposal, unit)
            "distribusi_bahasa_asli": {bahasa: int},            # PER PROPOSAL, bukan per unit
            "distribusi_format_asli": {format_asli: int},       # PER PROPOSAL, bukan per unit
            "proposal_tanpa_anotasi_sama_sekali": [proposal_id, ...],  # semua unit-nya TIDAK_ADA_ANOTASI

            "peringatan": [str, ...],   # record cacat (field hilang/tipe salah) dilaporkan di sini, bukan diam-diam dilewati
        }
    """
    peringatan: List[str] = []

    if not list_record:
        peringatan.append("list_record kosong; seluruh statistik dikembalikan sebagai nol/kosong.")
        return {
            "jumlah_proposal": 0,
            "jumlah_unit_total": 0,
            "jumlah_pelanggaran_per_kode": {},
            "rata_rata_pelanggaran_per_proposal": 0.0,
            "kode_instans_rendah": [],
            "jumlah_pelanggaran_per_kategori": {},
            "jumlah_pelanggaran_per_unit": {},
            "distribusi_status_ground_truth": {},
            "distribusi_bahasa_asli": {},
            "distribusi_format_asli": {},
            "proposal_tanpa_anotasi_sama_sekali": [],
            "peringatan": peringatan,
        }

    proposal_ids: Set[str] = set()
    hitung_kode: Counter = Counter()
    hitung_kategori: Counter = Counter()
    hitung_per_unit: Counter = Counter()
    hitung_status: Counter = Counter()
    bahasa_per_proposal: Dict[Any, Any] = {}
    format_per_proposal: Dict[Any, Any] = {}
    status_per_proposal: Dict[Any, List[str]] = defaultdict(list)

    for i, rec in enumerate(list_record):
        proposal_id = rec.get("proposal_id")
        unit_id = rec.get("unit_id")
        if proposal_id is None or unit_id is None:
            peringatan.append(
                f"Record indeks {i} tidak punya proposal_id/unit_id yang valid, dilewati dari statistik."
            )
            continue
        proposal_ids.add(proposal_id)

        status = rec.get("status_ground_truth", "TIDAK_DIKETAHUI")
        hitung_status[status] += 1
        status_per_proposal[proposal_id].append(status)

        kode_list = rec.get("kode_pelanggaran") or []
        if not isinstance(kode_list, list):
            peringatan.append(
                f"Record (proposal_id={proposal_id}, unit_id={unit_id}): 'kode_pelanggaran' "
                f"bertipe {type(kode_list).__name__} (bukan list), dilewati dari hitungan kode."
            )
            kode_list = []
        for kode in kode_list:
            hitung_kode[kode] += 1
            hitung_kategori[_kategori_dari_kode(kode)] += 1
            hitung_per_unit[unit_id] += 1

        if proposal_id not in bahasa_per_proposal:
            bahasa_per_proposal[proposal_id] = rec.get("bahasa_asli")
        if proposal_id not in format_per_proposal:
            format_per_proposal[proposal_id] = rec.get("format_asli")

    jumlah_proposal = len(proposal_ids)
    total_pelanggaran = sum(hitung_kode.values())
    rata_rata = (total_pelanggaran / jumlah_proposal) if jumlah_proposal else 0.0

    jumlah_pelanggaran_per_kode = dict(sorted(hitung_kode.items(), key=lambda kv: (-kv[1], kv[0])))
    kode_instans_rendah = sorted(
        kode for kode, jumlah in hitung_kode.items() if jumlah < ambang_instans_rendah
    )

    proposal_tanpa_anotasi = sorted(
        pid
        for pid, daftar_status in status_per_proposal.items()
        if daftar_status and all(s == "TIDAK_ADA_ANOTASI" for s in daftar_status)
    )
    if proposal_tanpa_anotasi:
        peringatan.append(
            "Proposal berikut sama sekali tidak punya anotasi ground truth (seluruh "
            "unit-nya berstatus TIDAK_ADA_ANOTASI), kemungkinan belum dianotasi: "
            + ", ".join(str(p) for p in proposal_tanpa_anotasi)
        )

    return {
        "jumlah_proposal": jumlah_proposal,
        "jumlah_unit_total": len(list_record),
        "jumlah_pelanggaran_per_kode": jumlah_pelanggaran_per_kode,
        "rata_rata_pelanggaran_per_proposal": round(rata_rata, 3),
        "kode_instans_rendah": kode_instans_rendah,
        "jumlah_pelanggaran_per_kategori": dict(
            sorted(hitung_kategori.items(), key=lambda kv: (-kv[1], kv[0]))
        ),
        "jumlah_pelanggaran_per_unit": dict(sorted(hitung_per_unit.items())),
        "distribusi_status_ground_truth": dict(
            sorted(hitung_status.items(), key=lambda kv: (-kv[1], kv[0]))
        ),
        "distribusi_bahasa_asli": dict(Counter(bahasa_per_proposal.values())),
        "distribusi_format_asli": dict(Counter(format_per_proposal.values())),
        "proposal_tanpa_anotasi_sama_sekali": proposal_tanpa_anotasi,
        "peringatan": peringatan,
    }


## Bagian F — Split Dev/Test

Modul ini membagi daftar proposal_id menjadi dua himpunan (dev, test) di
LEVEL PROPOSAL (bukan level unit — satu proposal tidak boleh punya
sebagian unit di dev dan sebagian lain di test, karena unit-unit dalam
satu proposal saling terkait/dibandingkan pada jalur JD). Pembagian
distratifikasi berdasarkan `bahasa_asli` supaya proporsi ID/EN di dev dan
test mirip, DENGAN PENGECUALIAN: proposal berbahasa Inggris di test set
dijamin minimal `minimum_per_bahasa["EN"]` (default 8) — kalau rasio umum
(`test_ratio`) tidak cukup menghasilkan itu, proporsi test untuk bahasa
EN dinaikkan melebihi rasio umum. Kalau jumlah proposal EN yang tersedia
bahkan kurang dari minimum itu sendiri, SEMUA proposal EN dialokasikan ke
test dan hal ini dilaporkan lewat `warnings.warn` (bukan gagal diam-diam),
karena minimum tidak akan tercapai.


In [ ]:
def split_dataset(
    list_proposal_id: List[str],
    metadata_per_proposal: Dict[str, Dict[str, Any]],
    test_ratio: float = 0.2,
    seed: int = 42,
    minimum_per_bahasa: Optional[Dict[str, int]] = None,
    kolom_bahasa: str = "bahasa_asli",
) -> Tuple[List[str], List[str]]:
    """Membagi proposal_id jadi dev/test, stratifikasi bahasa dgn jaminan minimum EN.

    Input:
        list_proposal_id: daftar proposal_id (level proposal, bukan unit).
            Duplikat dibuang otomatis (dgn peringatan) sebelum split.
        metadata_per_proposal: dict proposal_id -> metadata (minimal berisi
            key `kolom_bahasa`, mis. "ID"/"EN"). Proposal yang tidak
            ditemukan di sini atau tidak punya nilai `kolom_bahasa` yang
            valid dikelompokkan sbg bahasa "TIDAK_DIKETAHUI" (dgn peringatan),
            BUKAN menyebabkan proses gagal.
        test_ratio: proporsi standar ke test set per kelompok bahasa
            (default 0.2 = 20%). Proporsi AKTUAL bisa lebih tinggi untuk
            kelompok yang kena aturan `minimum_per_bahasa` (lihat di bawah).
        seed: seed RNG (Python `random.Random`) supaya split deterministik/
            dapat direproduksi untuk seed yang sama.
        minimum_per_bahasa: override manual jumlah minimum proposal per
            bahasa yang WAJIB masuk test set, mis. `{"EN": 8}` (default
            kalau None). Set ke `{}` untuk menonaktifkan jaminan minimum
            sama sekali (murni `test_ratio` stratifikasi biasa utk semua
            bahasa). Bahasa yang tidak disebut di sini tidak punya jaminan
            minimum (murni ikut `test_ratio`).
        kolom_bahasa: nama key bahasa di `metadata_per_proposal[pid]`
            (default "bahasa_asli").

    Output:
        (dev_ids, test_ids) — dua list proposal_id yang saling lepas,
        gabungannya = seluruh `list_proposal_id` (setelah duplikat
        dibuang), urutan mengikuti urutan asli `list_proposal_id`.

    Efek samping: memanggil `warnings.warn()` (BUKAN mengembalikan nilai
    tambahan, supaya kontrak return `(dev_ids, test_ids)` tetap sederhana)
    untuk kondisi yang perlu ditinjau manusia: duplikat proposal_id,
    proposal tanpa metadata bahasa, dan kasus minimum per-bahasa yang
    menaikkan proporsi test di atas `test_ratio` atau bahkan tidak
    tercapai sama sekali.

    Raises:
        ValueError: `test_ratio` di luar rentang (0, 1).
    """
    if not 0 < test_ratio < 1:
        raise ValueError(f"test_ratio harus di antara 0 dan 1 (eksklusif), diberikan: {test_ratio}")

    minimum_per_bahasa = dict(minimum_per_bahasa) if minimum_per_bahasa is not None else {"EN": 8}

    daftar_unik = list(dict.fromkeys(list_proposal_id))
    if len(daftar_unik) != len(list_proposal_id):
        warnings.warn(
            f"list_proposal_id mengandung {len(list_proposal_id) - len(daftar_unik)} "
            "proposal_id duplikat; duplikat dibuang (dipertahankan kemunculan pertama) "
            "sebelum split.",
            stacklevel=2,
        )

    kelompok: Dict[str, List[str]] = defaultdict(list)
    for pid in daftar_unik:
        info = metadata_per_proposal.get(pid)
        bahasa = info.get(kolom_bahasa) if info else None
        if not bahasa:
            bahasa = "TIDAK_DIKETAHUI"
            warnings.warn(
                f"proposal_id '{pid}' tidak punya metadata '{kolom_bahasa}' yang valid "
                "di `metadata_per_proposal`; dikelompokkan sbg bahasa 'TIDAK_DIKETAHUI' "
                "untuk keperluan stratifikasi.",
                stacklevel=2,
            )
        kelompok[bahasa].append(pid)

    rng = random.Random(seed)
    dev_ids: List[str] = []
    test_ids: List[str] = []

    for bahasa, anggota in kelompok.items():
        anggota_acak = anggota[:]
        rng.shuffle(anggota_acak)

        jumlah_test_standar = round(len(anggota_acak) * test_ratio)
        jumlah_minimum = minimum_per_bahasa.get(bahasa, 0)
        jumlah_test = max(jumlah_test_standar, min(jumlah_minimum, len(anggota_acak)))

        if jumlah_minimum > len(anggota_acak):
            warnings.warn(
                f"Bahasa '{bahasa}': hanya ada {len(anggota_acak)} proposal, kurang dari "
                f"minimum yang diminta di test set ({jumlah_minimum}). SEMUA proposal "
                f"bahasa ini ({len(anggota_acak)}) dialokasikan ke test; minimum TIDAK "
                "tercapai — pertimbangkan menambah data atau menurunkan `minimum_per_bahasa`.",
                stacklevel=2,
            )
        elif jumlah_test > jumlah_test_standar:
            warnings.warn(
                f"Bahasa '{bahasa}': rasio standar ({test_ratio:.0%}) hanya menghasilkan "
                f"{jumlah_test_standar} proposal di test, di bawah minimum {jumlah_minimum}. "
                f"Proporsi test untuk bahasa ini dinaikkan jadi {jumlah_test} proposal "
                f"(~{jumlah_test / len(anggota_acak):.0%}) supaya minimum terpenuhi.",
                stacklevel=2,
            )

        test_ids.extend(anggota_acak[:jumlah_test])
        dev_ids.extend(anggota_acak[jumlah_test:])

    # Urutkan kembali sesuai urutan asli list_proposal_id (bukan urutan
    # acak per-kelompok) supaya hasilnya deterministik & enak dibaca;
    # ini TIDAK mengubah proposal mana yang masuk dev/test, hanya urutannya.
    urutan_asli = {pid: i for i, pid in enumerate(daftar_unik)}
    dev_ids.sort(key=lambda pid: urutan_asli[pid])
    test_ids.sort(key=lambda pid: urutan_asli[pid])

    return dev_ids, test_ids


## Bagian G — Orkestrasi Pipeline & Output Akhir

Modul ini merangkai seluruh tahap sebelumnya (A: ekstraksi teks, B:
segmentasi unit, C: metadata JM, D: gabung ground truth, E: statistik,
F: split dev/test) jadi satu pipeline yang bisa dijalankan atas satu
folder proposal + satu workbook ground truth, menghasilkan:
- satu file JSONL (satu baris per (proposal_id, unit_id)) untuk dikonsumsi
  pipeline RAG, dan
- satu file log audit TEKS BIASA (bukan JSONL, bukan data) berisi seluruh
  warning/error per proposal, untuk ditinjau manusia.

Prinsip "satu proposal gagal, batch tetap lanjut" diterapkan di
`proses_folder_proposal`: kegagalan total satu file (mis. file corrupt)
DITANGKAP di situ, dicatat ke log, dan proses lanjut ke file berikutnya —
lihat komentar di dalam fungsi tsb untuk penjelasan kenapa `except
Exception` yang luas SENGAJA dipakai di titik itu saja.


In [ ]:
PathLike = Union[str, Path]


# ---------------------------------------------------------------------------
# Metadata proposal (sheet "Daftar_Proposal")
# ---------------------------------------------------------------------------


def muat_metadata_proposal(
    path_excel: PathLike,
    nama_sheet: str = "Daftar_Proposal",
    kolom_proposal_id: str = "proposal_id",
) -> Dict[str, Dict[str, Any]]:
    """Memuat metadata per proposal (bahasa_asli, dst.) dari sheet "Daftar_Proposal".

    Setiap baris sheet menjadi satu entri `{proposal_id: {nama_kolom: nilai, ...}}`
    — SELURUH kolom disertakan apa adanya (bukan cuma bahasa_asli), supaya
    fungsi ini tetap berguna kalau sheet Anda punya kolom metadata lain.

    PENTING (lihat juga catatan kalibrasi serupa di Bagian D): nama sheet &
    nama kolom proposal_id di sini adalah TEBAKAN berdasar deskripsi umum
    ("field bahasa_asli sudah ada di metadata proposal, sheet Daftar_Proposal")
    — saya tidak punya skema kolom literal sheet ini. Sesuaikan
    `nama_sheet`/`kolom_proposal_id` kalau berbeda di file Anda.

    Raises:
        GroundTruthError: sheet tidak bisa dibaca atau kolom proposal_id tidak ada.
    """
    path_excel = Path(path_excel)
    try:
        df = pd.read_excel(path_excel, sheet_name=nama_sheet, engine="openpyxl", dtype=str)
    except Exception as exc:
        raise GroundTruthError(
            f"Gagal membaca sheet '{nama_sheet}' dari '{path_excel.name}': {exc}"
        ) from exc

    if kolom_proposal_id not in df.columns:
        raise GroundTruthError(f"Kolom '{kolom_proposal_id}' tidak ditemukan di sheet '{nama_sheet}'.")

    hasil: Dict[str, Dict[str, Any]] = {}
    for _, baris in df.iterrows():
        pid = str(baris[kolom_proposal_id]).strip()
        if not pid or pid.lower() == "nan":
            continue
        hasil[pid] = baris.to_dict()
    return hasil


# ---------------------------------------------------------------------------
# Override status NA-01 (lihat catatan di Bagian A: keputusan akhir NA-01
# diambil di tahap ini, bukan di modul ekstraksi)
# ---------------------------------------------------------------------------


def _terapkan_override_na01(record_list: List[Dict[str, Any]], proposal_id: str, log: List[str]) -> None:
    """Meng-override `status_ground_truth` -> "NA-01" untuk SEMUA unit satu
    proposal, dipanggil ketika Bagian A mendeteksi `kemungkinan_hasil_pindai=True`
    pada file PDF-nya.

    Ini SENGAJA mengubah `record_list` in-place (dipanggil tepat sebelum
    `proses_satu_proposal` mengembalikan hasilnya). Rasional: kalau teks
    tidak berhasil diekstrak dengan andal dari file hasil pindai, evaluasi
    JP/JC/JD terhadap teks itu tidak bermakna, sehingga proposal ini lebih
    tepat ditandai NA-01 daripada memakai status dari ground truth apa
    adanya. Status ASLI (sebelum override) tetap dicatat ke log supaya
    keputusan ini bisa diaudit/dibatalkan manual kalau ternyata keliru
    (mis. PDF yang genuinely sedikit teksnya tapi bukan hasil scan).
    """
    for rec in record_list:
        status_asli = rec["status_ground_truth"]
        if status_asli != "NA-01":
            log.append(
                f"[{proposal_id}][{rec['unit_id']}] status_ground_truth di-override dari "
                f"'{status_asli}' menjadi 'NA-01' karena ekstraksi teks (Bagian A) mengindikasikan "
                "file ini kemungkinan hasil pindai/scan."
            )
        rec["status_ground_truth"] = "NA-01"
        rec["kode_pelanggaran"] = []
        catatan_tambahan = "Status di-override jadi NA-01: proposal terdeteksi kemungkinan hasil pindai/scan (Bagian A)."
        rec["catatan_ekstraksi"] = (
            f"{rec['catatan_ekstraksi']}; {catatan_tambahan}" if rec["catatan_ekstraksi"] else catatan_tambahan
        )


# ---------------------------------------------------------------------------
# Satu proposal end-to-end (A -> B -> C -> D)
# ---------------------------------------------------------------------------


def proses_satu_proposal(
    path_file: PathLike,
    proposal_id: str,
    gt_dataframe: pd.DataFrame,
    bahasa_asli: Optional[str] = None,
    peta_kolom: Optional[Dict[str, str]] = None,
) -> Tuple[List[Dict[str, Any]], List[str]]:
    """Memproses SATU file proposal dari Bagian A sampai D, hasilkan 16 record + log.

    Input:
        path_file: path ke file proposal (.docx/.pdf).
        proposal_id: ID proposal (biasanya nama file tanpa ekstensi).
        gt_dataframe: keluaran `muat_ground_truth()` (Bagian D).
        bahasa_asli: "ID"/"EN" dari metadata proposal, kalau ada.
        peta_kolom: diteruskan ke `gabungkan_teks_dan_ground_truth` (Bagian D).

    Output:
        (record_list, log_proposal) — 16 record (satu per unit, format
        Bagian D) dan list pesan log KHUSUS proposal ini (sudah diberi
        prefix "[proposal_id]"/"[proposal_id][unit_id]").

    Penanganan kegagalan (lihat juga docstring modul):
        - Kegagalan Bagian A (ekstraksi teks) atau Bagian B (segmentasi)
          DIBIARKAN merambat (raise) ke pemanggil, karena tanpa teks tidak
          ada apa pun yang bisa diproses secara bermakna untuk proposal
          ini — pemanggil batch (`proses_folder_proposal`) yang
          bertanggung jawab menangkapnya supaya batch tetap lanjut.
        - Kegagalan Bagian C (metadata JM) DITANGKAP DI SINI dan
          didegradasi jadi `metadata_jm = None` + catatan log, karena
          jalur JP/JC/JD masih bisa jalan tanpa metadata tata letak —
          kehilangan JM tidak seharusnya menggagalkan seluruh proposal.

    Raises:
        EkstraksiError: kegagalan Bagian A.
        (Exception lain dari Bagian B/D diteruskan apa adanya.)
    """
    log: List[str] = []

    hasil_ekstraksi = ekstrak_dokumen(path_file)
    log.extend(f"[{proposal_id}] {p}" for p in hasil_ekstraksi.get("peringatan", []))

    hasil_segmentasi = segmentasi_unit(hasil_ekstraksi, hasil_ekstraksi["format_asli"])
    log.extend(f"[{proposal_id}] {p}" for p in hasil_segmentasi.get("peringatan", []))

    try:
        hasil_metadata = ekstrak_metadata_dokumen(path_file)
        log.extend(f"[{proposal_id}] {p}" for p in hasil_metadata.get("peringatan", []))
    except EkstraksiError as exc:
        log.append(
            f"[{proposal_id}] Gagal ekstraksi metadata JM (Bagian C): {exc}. "
            "metadata_jm akan bernilai None untuk U00 proposal ini."
        )
        hasil_metadata = None

    record_list = gabungkan_teks_dan_ground_truth(
        hasil_segmentasi,
        gt_dataframe,
        proposal_id,
        metadata_jm=hasil_metadata,
        bahasa_asli=bahasa_asli,
        format_asli=hasil_ekstraksi["format_asli"],
        peta_kolom=peta_kolom,
    )

    for rec in record_list:
        if rec["catatan_ekstraksi"]:
            log.append(f"[{proposal_id}][{rec['unit_id']}] {rec['catatan_ekstraksi']}")

    if hasil_ekstraksi["format_asli"] == "pdf" and hasil_ekstraksi.get("kemungkinan_hasil_pindai"):
        _terapkan_override_na01(record_list, proposal_id, log)

    return record_list, log


# ---------------------------------------------------------------------------
# Batch: seluruh folder
# ---------------------------------------------------------------------------


def proses_folder_proposal(
    folder_proposal: PathLike,
    path_ground_truth_excel: PathLike,
    metadata_proposal: Optional[Dict[str, Dict[str, Any]]] = None,
    peta_kolom: Optional[Dict[str, str]] = None,
    pola_glob: str = "*.docx,*.pdf",
) -> Tuple[List[Dict[str, Any]], List[str]]:
    """Memproses SEMUA file proposal (.docx/.pdf campur) di satu folder.

    Input:
        folder_proposal: folder berisi file proposal. `proposal_id` diambil
            dari nama file TANPA ekstensi (mis. "P001.docx" -> proposal_id
            "P001") — pastikan skema penamaan file ini konsisten dengan
            proposal_id di ground truth. CATATAN: tahap redaksi PII terpisah
            TIDAK dipakai di implementasi ini (lihat catatan asumsi di
            docstring Bagian A/`ekstraksi_teks.py`) — isi file proposal
            TIDAK dianonimkan oleh pipeline ini.
        path_ground_truth_excel: workbook ground truth (dibaca sekali via
            `muat_ground_truth`, dipakai untuk semua proposal).
        metadata_proposal: dict proposal_id -> metadata (mis. dari
            `muat_metadata_proposal`), dipakai mengisi `bahasa_asli`. Kalau
            None, `bahasa_asli` semua proposal akan None.
        peta_kolom: diteruskan ke `muat_ground_truth`/`gabungkan_teks_dan_ground_truth`.
        pola_glob: pola glob file yang diproses, dipisah koma (default
            "*.docx,*.pdf").

    Output:
        (semua_record, semua_log) — gabungan record SELURUH proposal (siap
        ditulis lewat `tulis_output_jsonl`) dan gabungan log SELURUH
        proposal + pesan tingkat-batch (siap ditulis lewat `tulis_log_audit`).

    Raises:
        GroundTruthError: kalau workbook ground truth SENDIRI gagal dimuat
            (ini kegagalan tingkat BATCH, bukan tingkat proposal — tanpa
            ground truth, tidak ada satu proposal pun yang bisa diproses
            secara bermakna, jadi wajar dihentikan di sini, bukan di-skip
            per proposal).

    Kegagalan MEMPROSES SATU FILE PROPOSAL (docx/pdf corrupt, format tak
    dikenal, dll.) TIDAK menghentikan batch: ditangkap, dicatat ke log
    dengan detail error, lanjut ke file berikutnya. Exception generik
    (`Exception`, bukan cuma `EkstraksiError`) SENGAJA ditangkap di titik
    ini SATU-SATUNYA supaya bug tak terduga pada satu file pun tidak
    menggagalkan seluruh batch pemrosesan puluhan/ratusan proposal lain —
    di luar titik ini (Bagian A-F, fungsi single-proposal di atas), error
    dibiarkan merambat normal supaya mudah dites/didebug.
    """
    folder_proposal = Path(folder_proposal)
    metadata_proposal = metadata_proposal or {}

    try:
        gt_dataframe = muat_ground_truth(path_ground_truth_excel, peta_kolom=peta_kolom)
    except GroundTruthError as exc:
        raise GroundTruthError(
            f"Gagal memuat ground truth dari '{path_ground_truth_excel}', seluruh batch "
            f"dibatalkan (tanpa ground truth tidak ada yang bisa diproses bermakna): {exc}"
        ) from exc

    semua_record: List[Dict[str, Any]] = []
    semua_log: List[str] = [
        f"Ringkasan validasi ground truth: {gt_dataframe.attrs.get('ringkasan_validasi')}",
    ]

    daftar_file = sorted(
        {p for pola in pola_glob.split(",") for p in folder_proposal.glob(pola.strip())}
    )
    if not daftar_file:
        semua_log.append(
            f"Tidak ada file proposal ditemukan di folder '{folder_proposal}' (pola: {pola_glob})."
        )

    for path_file in daftar_file:
        proposal_id = path_file.stem
        info_proposal = metadata_proposal.get(proposal_id, {})
        bahasa_asli = info_proposal.get("bahasa_asli")

        try:
            record_list, log_proposal = proses_satu_proposal(
                path_file, proposal_id, gt_dataframe, bahasa_asli=bahasa_asli, peta_kolom=peta_kolom
            )
        except Exception as exc:  # noqa: BLE001 -- lihat penjelasan di docstring
            semua_log.append(
                f"[{proposal_id}] GAGAL DIPROSES TOTAL ({type(exc).__name__}): {exc}. "
                "Proposal ini TIDAK menghasilkan record apa pun di output JSONL; "
                "perlu ditinjau manual (file corrupt? format tak terduga?)."
            )
            continue

        semua_record.extend(record_list)
        semua_log.extend(log_proposal)

    return semua_record, semua_log


# ---------------------------------------------------------------------------
# Penulisan output akhir
# ---------------------------------------------------------------------------


def tulis_output_jsonl(record_list: List[Dict[str, Any]], path_output: PathLike) -> None:
    """Menulis `record_list` ke file JSONL, satu baris JSON per (proposal_id, unit_id).

    Field per record mengikuti keluaran `gabungkan_teks_dan_ground_truth`
    (Bagian D): proposal_id, unit_id, bahasa_asli, format_asli, teks_unit,
    metadata_jm, status_ground_truth, kode_pelanggaran, dasar_pedoman,
    bukti, keyakinan, anotator, status_adjudikasi, catatan_ekstraksi.
    File ini yang dikonsumsi pipeline RAG.
    """
    path_output = Path(path_output)
    with path_output.open("w", encoding="utf-8") as f:
        for rec in record_list:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def tulis_log_audit(daftar_log: List[str], path_log: PathLike) -> None:
    """Menulis seluruh pesan log ke file TEKS BIASA (bukan JSONL, bukan data
    untuk pipeline RAG) — satu pesan per baris, untuk ditinjau manusia
    (mis. proposal mana yang NA-01, unit mana yang gagal terdeteksi
    headingnya, kolom ground truth mana yang bermasalah, dst).
    """
    path_log = Path(path_log)
    with path_log.open("w", encoding="utf-8") as f:
        for baris in daftar_log:
            f.write(str(baris) + "\n")


## Contoh Pemanggilan End-to-End di Colab

Alur pemakaian:
1. Mount Google Drive (atau upload manual lewat panel Files di sebelah kiri Colab).
2. Siapkan **satu folder** berisi file proposal (`.docx`/`.pdf`, nama file = proposal_id, mis. `P001.docx`)
   dan **satu file** workbook ground truth (`.xlsx`, sheet `Anotasi` + `Daftar_Proposal`).
3. Sesuaikan variabel path di sel berikut, lalu jalankan.

Kalau nama kolom di workbook ground truth Anda berbeda dari default (lihat catatan kalibrasi Bagian D
di atas), berikan `peta_kolom=...` ke `proses_folder_proposal(...)`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# --- SESUAIKAN 3 PATH INI ---
FOLDER_PROPOSAL = "/content/drive/MyDrive/comply-proposal/proposal"        # folder berisi .docx/.pdf
PATH_GROUND_TRUTH = "/content/drive/MyDrive/comply-proposal/ground_truth.xlsx"  # workbook ground truth
FOLDER_OUTPUT = "/content/drive/MyDrive/comply-proposal/output"            # folder hasil JSONL + log
# -----------------------------

Path(FOLDER_OUTPUT).mkdir(parents=True, exist_ok=True)


In [ ]:
# Muat metadata proposal (bahasa_asli, dst.) dari sheet "Daftar_Proposal"
try:
    metadata_proposal = muat_metadata_proposal(PATH_GROUND_TRUTH)
except GroundTruthError as exc:
    print(f"Peringatan: gagal memuat sheet Daftar_Proposal ({exc}); bahasa_asli akan kosong untuk semua proposal.")
    metadata_proposal = {}

# Proses seluruh folder proposal (satu file gagal TIDAK menghentikan batch, lihat log)
try:
    semua_record, semua_log = proses_folder_proposal(
        FOLDER_PROPOSAL, PATH_GROUND_TRUTH, metadata_proposal=metadata_proposal
    )
except GroundTruthError as exc:
    raise SystemExit(f"Batch dibatalkan: {exc}")

path_jsonl = Path(FOLDER_OUTPUT) / "dataset_uji.jsonl"
path_log = Path(FOLDER_OUTPUT) / "log_audit.txt"
tulis_output_jsonl(semua_record, path_jsonl)
tulis_log_audit(semua_log, path_log)

print(f"Selesai: {len(semua_record)} record ditulis ke {path_jsonl}")
print(f"Log audit ({len(semua_log)} baris) ditulis ke {path_log}")


### Statistik dataset (Bagian E)

In [ ]:
statistik = hitung_statistik(semua_record)
print(json.dumps(statistik, indent=2, ensure_ascii=False))


### Split dev/test (Bagian F)

`minimum_per_bahasa={"EN": 8}` (default) menjamin proposal EN di test set minimal 8 buah — sesuaikan
kalau komposisi dataset Anda (mis. 42 ID / 18 EN) butuh angka lain.

In [ ]:
daftar_proposal_id = sorted({rec["proposal_id"] for rec in semua_record})
dev_ids, test_ids = split_dataset(daftar_proposal_id, metadata_proposal)

print(f"Dev: {len(dev_ids)} proposal | Test: {len(test_ids)} proposal")
print("dev :", dev_ids)
print("test:", test_ids)


### Pratinjau beberapa baris pertama `dataset_uji.jsonl`

In [ ]:
import itertools

with open(path_jsonl, encoding="utf-8") as f:
    for baris in itertools.islice(f, 5):
        rec = json.loads(baris)
        print(rec["proposal_id"], rec["unit_id"], "|", rec["status_ground_truth"], "|", rec["kode_pelanggaran"])
